In [ ]:
# Features: 4 kHz, 5 s frames (50% overlap), 0-1 Hz removed, max-abs -> 3 x 64 x 40 (mel, CQT, MFCC), float16.
# Non-COPD classes are augmented up to the COPD frame count. Finished stages are skipped.
import os, time, json, numpy as np, pandas as pd, librosa
from skimage.transform import resize

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# ================= EDIT ONLY THIS BLOCK =================
VARIANT = "orig"     # "orig" = original features (already built: nothing is recomputed)
                     # "win64" = mel + MFCC window 1024 -> 256 samples (256 ms -> 64 ms) for the window side test (trial 21)
TOL = 5e-3           # allowed CQT difference when checking that "win64" rebuilt exactly the same frames
# =======================================================
N_FFT = {"orig": 1024, "win64": 256}[VARIANT]

BASE = "/content/drive/MyDrive/Data"
PROJ = os.path.join(BASE, "SeedPaper_CCT")
FEAT0 = os.path.join(PROJ, "features")                                   # original features (never overwritten)
FEAT = FEAT0 if VARIANT == "orig" else os.path.join(PROJ, f"features_{VARIANT}")
CKPT, RES = (os.path.join(PROJ, d) for d in ("checkpoints", "results"))
for d in (FEAT, CKPT, RES): os.makedirs(d, exist_ok=True)

SR, FL, HOP = 4000, 20000, 10000            # ৪ kHz, ৫ সেকেন্ড, ৫০% overlap
CLASSES = ["COPD", "Healthy", "Asthma", "Pneumonia", "URTI", "Bronchiectasis", "Bronchiolitis"]
rng = np.random.default_rng(42)

json.dump(dict(sampling_rate=SR, frame_seconds=5, overlap=0.5, baseline_wander_cutoff_hz=1,
               normalization="max-abs", mel=dict(n_fft=N_FFT, hop=524, window="hamming", n_mels=64),
               cqt=dict(n_bins=64, hop=512, fmin=32.7, bins_per_octave=12, window="hamming"),
               mfcc=dict(n_mfcc=30, n_mels=30, n_fft=N_FFT, hop=512, window="hamming"),
               augmentation=dict(time_stretch=[0.85, 1.15], pitch_semitones=[-3, 3], noise_factor=[0.001, 0.01],
                                 target="COPD frame count, only non-COPD classes"),
               model_input="3 channel (mel, CQT, MFCC) resized to 64x40", variant=VARIANT),
          open(os.path.join(PROJ, "pipeline_config.json" if VARIANT == "orig" else f"pipeline_config_{VARIANT}.json"), "w"), indent=2)

# ---------- ধাপ ০: recording-এর তালিকা (১৬৬৭টা) ----------
seg = pd.read_csv(os.path.join(BASE, "segment_metadata.csv"))
v3 = pd.read_csv(os.path.join(BASE, "final_split_dataset_v3.csv"))
v3r = v3[v3["source"] != "AUGMENTED"][["segment_id", "pkey", "fold"]]
rec = (seg.merge(v3r, on="segment_id").drop_duplicates("original_filepath")
          [["original_filepath", "source", "unified_label", "pkey", "fold"]].reset_index(drop=True))
print(f"recording: {len(rec)} (প্রত্যাশিত 1667)")

# ---------- ধাপ ১-৪: preprocessing ----------
def preprocess_recording(path):
    y, sr = librosa.load(path, sr=None, mono=True)
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)           # ১. ৪ kHz
    if len(y) < FL:
        y = np.pad(y, (0, FL - len(y)))
    n = 1 + (len(y) - FL) // HOP                                    # ২. ৫s frame, ৫০% overlap
    k = int(np.ceil(1.0 * FL / SR))                                 # ১ Hz-এর DFT index (= 5)
    frames = []
    for p in range(n):
        x = y[p * HOP: p * HOP + FL].astype(np.float64)
        X = np.fft.rfft(x); X[:k + 1] = 0                           # ৩. ০–১ Hz বাদ (baseline wander)
        x = np.fft.irfft(X, n=FL)
        m = np.max(np.abs(x))
        frames.append((x / m if m > 0 else x).astype(np.float32))   # ৪. max-abs normalize
    return np.stack(frames)

# ---------- ধাপ ৬-৮: TTFFS (mel, CQT, MFCC) ----------
def _fit(a, shape=(64, 40)):
    a = (a - a.min()) / (a.max() - a.min() + 1e-8)
    if a.shape != shape:
        a = resize(a, shape, order=1, anti_aliasing=False)
    return a.astype(np.float32)

def ttffs(x):
    mel = librosa.power_to_db(librosa.feature.melspectrogram(
        y=x, sr=SR, n_fft=N_FFT, hop_length=524, win_length=N_FFT, window="hamming", n_mels=64), ref=np.max)
    cqt = librosa.power_to_db(np.abs(librosa.cqt(
        x, sr=SR, hop_length=512, fmin=32.7, n_bins=64, bins_per_octave=12, window="hamming")) ** 2, ref=np.max)
    mfcc = librosa.feature.mfcc(y=x, sr=SR, n_mfcc=30, n_mels=30, n_fft=N_FFT,
                                hop_length=512, win_length=N_FFT, window="hamming")
    return mel, cqt, mfcc

def to_tensor(x):                                                   # (3, 64, 40)
    m, c, f = ttffs(x)
    return np.stack([_fit(m), _fit(c), _fit(f)]).astype(np.float16)

print("TFR মাপ:", [a.shape for a in ttffs(np.random.randn(FL).astype(np.float32))],
      "(প্রত্যাশিত: (64,39) (64,40) (30,40))")

# ---------- ধাপ ৫: augmentation ----------
def augment(x):
    x = x.copy()
    choice = rng.choice(["stretch", "pitch", "noise", "combo"])
    if choice in ("stretch", "combo"):
        x = librosa.effects.time_stretch(x, rate=rng.uniform(0.85, 1.15))
    if choice in ("pitch", "combo"):
        x = librosa.effects.pitch_shift(x, sr=SR, n_steps=rng.uniform(-3, 3))
    if choice in ("noise", "combo"):
        x = x + rng.uniform(0.001, 0.01) * rng.standard_normal(len(x))
    x = x[:FL] if len(x) >= FL else np.pad(x, (0, FL - len(x)))
    m = np.max(np.abs(x))
    return (x / m if m > 0 else x).astype(np.float32)

def check_same_frames(new, old_path, what):
    """VARIANT != orig: the CQT channel does not depend on N_FFT, so it must equal the original one frame by frame.
       A mismatch means a different random augmentation (or library version): stop instead of training on other data."""
    old = np.load(old_path, mmap_mode="r")
    if len(old) != len(new): raise RuntimeError(f"{what}: {len(new)} frames vs {len(old)} in the original")
    d = float(np.max(np.abs(np.asarray(old[:, 1], np.float32) - new[:, 1].astype(np.float32))))
    if d > TOL: raise RuntimeError(f"{what}: CQT channel differs from the original (max diff {d:.4f}); features_{VARIANT} is NOT the same data")
    print(f"   ✔ {what}: CQT channel identical to the original (max diff {d:.1e})")

# ---------- Stage A: আসল frame ----------
fa = os.path.join(FEAT, "feat_orig.npy")
cols = ["rec_idx", "label", "source", "pkey", "fold"]
if not os.path.exists(fa):
    feats, meta, waves, wmeta, bad = [], [], [], [], 0
    t0 = time.time()
    for i in range(len(rec)):
        r = rec.iloc[i]
        try:
            fr = preprocess_recording(r["original_filepath"])
        except Exception as e:
            bad += 1; print("⚠️ পড়া যায়নি:", r["original_filepath"], type(e).__name__); continue
        row = (i, r["unified_label"], r["source"], r["pkey"], int(r["fold"]))
        for x in fr:
            feats.append(to_tensor(x)); meta.append(row)
            if r["unified_label"] != "COPD" and VARIANT == "orig":
                waves.append(x); wmeta.append(row)
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(rec)} recording | {len(feats)} frame | {(time.time()-t0)/60:.1f} মিনিট")
    feats = np.stack(feats)
    if VARIANT == "orig":
        np.save(os.path.join(FEAT, "wave_minority.npy"), np.stack(waves))
        pd.DataFrame(wmeta, columns=cols).to_csv(os.path.join(FEAT, "wave_meta.csv"), index=False)
    else:
        check_same_frames(feats, os.path.join(FEAT0, "feat_orig.npy"), "Stage A")
    pd.DataFrame(meta, columns=cols).to_csv(os.path.join(FEAT, "meta_orig.csv"), index=False)
    np.save(fa, feats)                      # সবশেষে সেভ: এটা থাকা মানে Stage A সম্পূর্ণ
    print(f"✅ Stage A: {len(feats)} আসল frame, {bad}টা recording ব্যর্থ")

# ---------- Stage B: প্রতি non-COPD ক্লাসকে COPD-র frame সংখ্যায় আনা ----------
mo = pd.read_csv(os.path.join(FEAT, "meta_orig.csv"))
counts = mo["label"].value_counts(); target = int(counts.max())
print("আসল frame সংখ্যা:\n" + counts.to_string() + f"\nTarget (COPD): {target}")
waves = np.load(os.path.join(FEAT0, "wave_minority.npy")); wm = pd.read_csv(os.path.join(FEAT0, "wave_meta.csv"))
for cls in CLASSES[1:]:
    fp = os.path.join(FEAT, f"feat_aug_{cls}.npy")
    st = os.path.join(FEAT, f"rng_state_after_{cls}.json")           # lets a stopped run resume with the same random sequence
    if os.path.exists(fp):
        if os.path.exists(st): rng.bit_generator.state = json.load(open(st))
        print(f"⏭️ {cls} আগেই আছে"); continue
    need = target - int(counts.get(cls, 0))
    picks = rng.choice(np.where(wm["label"].values == cls)[0], size=need, replace=True)
    if VARIANT != "orig":                                            # same frames picked as in the original run?
        orig_meta = pd.read_csv(os.path.join(FEAT0, f"meta_aug_{cls}.csv"))
        if not (wm.iloc[picks]["rec_idx"].values == orig_meta["rec_idx"].values).all():
            raise RuntimeError(f"{cls}: random picks differ from the original run; stop (do not delete files in features/)")
    out = np.zeros((need, 3, 64, 40), dtype=np.float16)
    t0 = time.time()
    for n, j in enumerate(picks):
        out[n] = to_tensor(augment(waves[j]))
        if (n + 1) % 2000 == 0:
            print(f"  {cls}: {n+1}/{need} ({(time.time()-t0)/60:.1f} মিনিট)")
    if VARIANT != "orig":
        check_same_frames(out, os.path.join(FEAT0, f"feat_aug_{cls}.npy"), cls)
    wm.iloc[picks].reset_index(drop=True).to_csv(os.path.join(FEAT, f"meta_aug_{cls}.csv"), index=False)
    np.save(fp, out)
    json.dump(rng.bit_generator.state, open(st, "w"))
    print(f"✅ {cls}: {need}টা augmented ({(time.time()-t0)/60:.1f} মিনিট)")
print("🎉 features সম্পূর্ণ → ফোল্ডার:", FEAT)

In [ ]:
# Split check: every patient (pkey) and every recording must sit in exactly one fold (basis of the patient-disjoint trial 28).
import os, pandas as pd

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/Data"

v = pd.read_csv(os.path.join(BASE, "final_split_dataset_v3.csv"))
v = v[v["source"] != "AUGMENTED"]

# প্রতিটা pkey-র fold একটাই?
print((v.groupby("pkey")["fold"].nunique() > 1).sum(), "টা pkey একাধিক fold-এ")

# প্রতিটা recording-এর fold একটাই?
seg = pd.read_csv(os.path.join(BASE, "segment_metadata.csv"))
m = seg.merge(v[["segment_id", "pkey", "fold"]], on="segment_id")
print((m.groupby("original_filepath")["fold"].nunique() > 1).sum(), "টা recording একাধিক fold-এ")

# dataset অনুযায়ী pkey ও ফাইল সংখ্যা
print(m.groupby("source").agg(files=("original_filepath", "nunique"),
                              pkeys=("pkey", "nunique")))

In [ ]:
# Figure: waveform, mel, CQT and MFCC of one recording per class, plus the 3-channel model input.
import os, numpy as np, pandas as pd, librosa
import matplotlib.pyplot as plt
from skimage.transform import resize

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Data"
RES = os.path.join(BASE, "SeedPaper_CCT", "results"); os.makedirs(RES, exist_ok=True)

SR, FL, HOP = 4000, 20000, 10000
CLASSES = ["Healthy", "COPD", "Asthma", "Pneumonia", "URTI", "Bronchiectasis", "Bronchiolitis"]
SEED = 7              # অন্য উদাহরণ দেখতে এই সংখ্যা বদলাও
SHOW_FUSION = True    # True = শেষ কলামে ৩-channel model input | False = শুধু seed paper-এর মতো ৪ কলাম

# ---- features সেলের হুবহু একই preprocessing ও TTFFS ----
def preprocess_recording(path):
    y, sr = librosa.load(path, sr=None, mono=True)
    if sr != SR: y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    if len(y) < FL: y = np.pad(y, (0, FL - len(y)))
    n = 1 + (len(y) - FL) // HOP; k = int(np.ceil(1.0 * FL / SR)); frames = []
    for p in range(n):
        x = y[p * HOP: p * HOP + FL].astype(np.float64)
        X = np.fft.rfft(x); X[:k + 1] = 0
        x = np.fft.irfft(X, n=FL); m = np.max(np.abs(x))
        frames.append((x / m if m > 0 else x).astype(np.float32))
    return np.stack(frames)

def ttffs(x):
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=x, sr=SR, n_fft=1024, hop_length=524, win_length=1024,
                                                             window="hamming", n_mels=64), ref=np.max)
    cqt = librosa.power_to_db(np.abs(librosa.cqt(x, sr=SR, hop_length=512, fmin=32.7, n_bins=64,
                                                 bins_per_octave=12, window="hamming")) ** 2, ref=np.max)
    mfcc = librosa.feature.mfcc(y=x, sr=SR, n_mfcc=30, n_mels=30, n_fft=1024, hop_length=512,
                                win_length=1024, window="hamming")
    return mel, cqt, mfcc

def fit(a, shape=(64, 40)):                       # to_tensor-এর হুবহু একই ধাপ
    a = (a - a.min()) / (a.max() - a.min() + 1e-8)
    return resize(a, shape, order=1, anti_aliasing=False) if a.shape != shape else a

# ---- প্রতি ক্লাস থেকে একটা রেকর্ডিং, মাঝের ৫ সেকেন্ডের frame ----
seg = pd.read_csv(os.path.join(BASE, "segment_metadata.csv"))
rec = seg.drop_duplicates("original_filepath")[["original_filepath", "source", "unified_label"]]
rng = np.random.default_rng(SEED)

ex = []
for cls in CLASSES:
    pool = rec[rec["unified_label"] == cls].sample(frac=1, random_state=int(rng.integers(10**6)))
    for _, r in pool.iterrows():
        try:
            fr = preprocess_recording(r["original_filepath"]); break
        except Exception:
            continue
    x = fr[len(fr) // 2]
    ex.append((cls, r["source"], x) + ttffs(x))

# ---- ছবি ----
ncol = 5 if SHOW_FUSION else 4
fig, axes = plt.subplots(len(ex), ncol, figsize=(4.2 * ncol, 2.55 * len(ex)), dpi=130)
titles = ["Waveform (after preprocessing)", "Mel spectrogram (64 x 39)", "CQT spectrogram (64 x 40)", "MFCC (30 x 40)",
          "CCT input, 3 channels\n(R = mel, G = CQT, B = MFCC), 64 x 40"]
t = np.arange(FL) / SR
for i, (cls, src, x, mel, cqt, mfcc) in enumerate(ex):
    ax = axes[i]
    ax[0].plot(t, x, lw=0.4, color="#1f4e9c"); ax[0].set_xlim(0, 5); ax[0].set_ylim(-1.05, 1.05)
    ax[1].imshow(mel, origin="lower", aspect="auto", extent=[0, 5, 0, 64], cmap="viridis")
    ax[2].imshow(cqt, origin="lower", aspect="auto", extent=[0, 5, 0, 64], cmap="viridis")
    ax[3].imshow(mfcc, origin="lower", aspect="auto", extent=[0, 5, 0, 30], cmap="viridis")
    ax[0].set_ylabel(f"{cls}\n({src})", fontsize=10, fontweight="bold")
    ax[1].set_ylabel("Mel band", fontsize=8); ax[2].set_ylabel("CQT bin", fontsize=8); ax[3].set_ylabel("MFCC coeff.", fontsize=8)
    if SHOW_FUSION:
        ax[4].imshow(np.stack([fit(mel), fit(cqt), fit(mfcc)], axis=-1), origin="lower", aspect="auto", extent=[0, 5, 0, 64])
        ax[4].set_ylabel("Freq. axis (resized to 64)", fontsize=7)
    for j in range(ncol):
        ax[j].tick_params(labelsize=7)
        if i == 0: ax[j].set_title(titles[j], fontsize=9.5)
        if i == len(ex) - 1: ax[j].set_xlabel("Time (s)", fontsize=8)
fig.suptitle("Respiratory sound -> time-frequency representations (one example per class; 5 s frame, 4 kHz)", fontsize=13, y=1.0)
fig.tight_layout()
out = os.path.join(RES, "ttffs_examples.png"); fig.savefig(out, bbox_inches="tight"); plt.show()
print("✅ ছবি সেভ হয়েছে:", out)

In [ ]:
# Replication of the seed-paper protocol: CCT, frame-level stratified 5-fold, Adam lr 0.005, batch 128, final-epoch model.
# Already run; finished folds are skipped.
import os, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Data"; PROJ = os.path.join(BASE, "SeedPaper_CCT")
FEAT, CKPT, RES = (os.path.join(PROJ, d) for d in ("features", "checkpoints", "results"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for _v in ("Xg", "yg", "DATA"):          # free a previous copy (~1 GB on the GPU) before loading again
    globals().pop(_v, None)
if torch.cuda.is_available(): torch.cuda.empty_cache()

SPLIT_MODE = "paper"      # "paper" = সব frame মিলিয়ে stratified random 5-fold | "patient" = patient-disjoint
FOLDS = [0]               # পূর্ণ run: [0, 1, 2, 3, 4]
EPOCHS = 20               # প্রথমে ছোট পরীক্ষা; ঠিক থাকলে 250 (seed paper)
LR, BATCH = 0.005, 128    # seed paper Table IV
TAG = f"cct_{SPLIT_MODE}"
CLASSES = ["COPD", "Healthy", "Asthma", "Pneumonia", "URTI", "Bronchiectasis", "Bronchiolitis"]
LABEL_MAP = {c: i for i, c in enumerate(CLASSES)}

# ---------- ডেটা লোড (আসল + augmented) ----------
pX = [np.load(os.path.join(FEAT, "feat_orig.npy"))]
mo = pd.read_csv(os.path.join(FEAT, "meta_orig.csv")); mo["aug"] = 0; pM = [mo]
for c in CLASSES[1:]:
    p = os.path.join(FEAT, f"feat_aug_{c}.npy")
    if os.path.exists(p):
        pX.append(np.load(p)); m = pd.read_csv(os.path.join(FEAT, f"meta_aug_{c}.csv")); m["aug"] = 1; pM.append(m)
X = np.concatenate(pX); M = pd.concat(pM, ignore_index=True)
y = M["label"].map(LABEL_MAP).values
Xg = torch.from_numpy(X).to(device); yg = torch.from_numpy(y).long().to(device)   # সব GPU-তে (float16)
print(f"মোট frame: {len(y)} | আসল {int((M['aug']==0).sum())} + augmented {int((M['aug']==1).sum())}")

# ---------- split ----------
if SPLIT_MODE == "paper":
    _splits = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(len(y)), y))
    def get_split(f): return _splits[f]
else:
    def get_split(f):
        tr = np.where(M["fold"].values != f)[0]
        te = np.where((M["fold"].values == f) & (M["aug"].values == 0))[0]   # test-এ শুধু আসল frame
        return tr, te

# ---------- CCT (৩×৬৪×৪০ ইনপুট) ----------
class ConvTokenizer(nn.Module):
    def __init__(self, c=3, e=64):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(c, e, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(3, 2, 1),
                                 nn.Conv2d(e, e, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(3, 2, 1))
    def forward(self, x):
        return self.net(x).flatten(2).transpose(1, 2)               # (B, 160 token, e)

class Block(nn.Module):
    def __init__(self, e, h, r=2, d=0.1):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(e), nn.LayerNorm(e)
        self.attn = nn.MultiheadAttention(e, h, dropout=d, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(e, e * r), nn.GELU(), nn.Dropout(d), nn.Linear(e * r, e), nn.Dropout(d))
    def forward(self, x):
        h = self.n1(x); x = x + self.attn(h, h, h)[0]
        return x + self.mlp(self.n2(x))

class CCT(nn.Module):
    def __init__(self, nc=7, e=64, layers=4, heads=4):
        super().__init__()
        self.tok = ConvTokenizer(3, e)
        self.blocks = nn.ModuleList([Block(e, heads) for _ in range(layers)])
        self.pool = nn.Linear(e, 1); self.fc = nn.Linear(e, nc)
    def forward(self, x):
        x = self.tok(x)
        for b in self.blocks: x = b(x)
        w = torch.softmax(self.pool(x), dim=1)
        return self.fc((w.transpose(1, 2) @ x).squeeze(1))
NPARAM = sum(p.numel() for p in CCT().parameters()); print("CCT parameter:", NPARAM)

@torch.no_grad()
def predict(model, idx_t):
    model.eval()
    return torch.cat([F.softmax(model(Xg[idx_t[i:i+2048]].float()), 1).cpu() for i in range(0, len(idx_t), 2048)]).numpy()

def evaluate(model, idx_t):
    P = predict(model, idx_t); yt = y[idx_t.cpu().numpy()]
    return float(-np.log(P[np.arange(len(yt)), yt] + 1e-9).mean()), float((P.argmax(1) == yt).mean())

# ---------- মাপ (seed paper-এর সূত্র: TP, TN, FP, FN) ----------
def metrics_table(yy, pp):
    cm = confusion_matrix(yy, pp, labels=list(range(7))); N = cm.sum(); rows = []
    for c, name in enumerate(CLASSES):
        TP = cm[c, c]; FN = cm[c].sum() - TP; FP = cm[:, c].sum() - TP; TN = N - TP - FN - FP
        se, sp, pr = TP / max(TP + FN, 1), TN / max(TN + FP, 1), TP / max(TP + FP, 1)
        rows.append((name, (TP + TN) / N, se, sp, pr, 2 * pr * se / max(pr + se, 1e-9), int(cm[c].sum())))
    t = pd.DataFrame(rows, columns=["class", "acc_1vsRest", "sensitivity", "specificity", "precision", "F1", "n"]).set_index("class")
    o = dict(accuracy=float(np.trace(cm) / N), majority_acc=float(cm.sum(1).max() / N), macro_f1=float(t["F1"].mean()),
             sensitivity=float(t["sensitivity"].mean()), specificity=float(t["specificity"].mean()),
             precision=float(t["precision"].mean()), n=int(N))
    return cm, t, o

def save_cm_png(cm, path, title):
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=150)
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(7)); ax.set_xticklabels(CLASSES, rotation=40, ha="right")
    ax.set_yticks(range(7)); ax.set_yticklabels(CLASSES)
    for i in range(7):
        for j in range(7):
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title, fontsize=9)
    plt.colorbar(im); fig.tight_layout(); fig.savefig(path); plt.close(fig)

SUMMARY = []
def save_report(P, title, name):
    cm, t, o = metrics_table(P["y"].values, P["pred"].values)
    line = (f"{title} ({o['n']} frame): accuracy {o['accuracy']:.4f} (সবচেয়ে বড় ক্লাস সবসময় বললে {o['majority_acc']:.4f}) | "
            f"macro-F1 {o['macro_f1']:.4f} | sensitivity {o['sensitivity']:.4f} | specificity {o['specificity']:.4f} | precision {o['precision']:.4f}")
    print("\n" + line); print(t.round(3).to_string())
    t.round(4).to_csv(os.path.join(RES, f"metrics_{TAG}_{name}.csv"))
    save_cm_png(cm, os.path.join(RES, f"confusion_{TAG}_{name}.png"), title)
    SUMMARY.append(line)

# ---------- training (প্রতি ১০ epoch-এ checkpoint) ----------
def run_fold(f):
    tr, te = get_split(f)
    tr_i, va_i = train_test_split(tr, test_size=0.10, stratify=y[tr], random_state=42)   # ১০% validation
    T = lambda a: torch.as_tensor(a, device=device)
    tr_t, va_t, te_t = T(tr_i), T(va_i), T(te)
    torch.manual_seed(42 + f)
    model = CCT().to(device); opt = torch.optim.Adam(model.parameters(), lr=LR)
    hist, start = [], 0
    ck = os.path.join(CKPT, f"ckpt_{TAG}_fold{f}.pt")
    if os.path.exists(ck):
        s = torch.load(ck, map_location=device, weights_only=False)
        model.load_state_dict(s["m"]); opt.load_state_dict(s["o"]); hist, start = s["h"], s["e"]
        print(f"  ↩️ epoch {start} থেকে চলবে")
    for ep in range(start, EPOCHS):
        t0 = time.time(); model.train()
        perm = tr_t[torch.randperm(len(tr_t), device=device)]; tot, corr = 0.0, 0
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]; xb, yb = Xg[b].float(), yg[b]
            opt.zero_grad(); out = model(xb); loss = F.cross_entropy(out, yb); loss.backward(); opt.step()
            tot += loss.item() * len(b); corr += (out.argmax(1) == yb).sum().item()
        vl, va = evaluate(model, va_t)
        hist.append(dict(epoch=ep + 1, train_loss=tot / len(perm), train_acc=corr / len(perm), val_loss=vl, val_acc=va))
        if ep == start or (ep + 1) % 10 == 0:
            h = hist[-1]
            print(f"  fold {f} | epoch {ep+1}/{EPOCHS} ({time.time()-t0:.0f}s/epoch) | train loss {h['train_loss']:.3f} acc {h['train_acc']:.3f} | val loss {h['val_loss']:.3f} acc {h['val_acc']:.3f}")
        if (ep + 1) % 10 == 0:
            torch.save({"m": model.state_dict(), "o": opt.state_dict(), "h": hist, "e": ep + 1}, ck)
    pd.DataFrame(hist).to_csv(os.path.join(RES, f"hist_{TAG}_fold{f}.csv"), index=False)
    return te, predict(model, te_t)

for f in FOLDS:
    pp = os.path.join(RES, f"pred_{TAG}_fold{f}.csv")
    if os.path.exists(pp):
        print(f"⏭️ fold {f} আগেই সম্পন্ন"); continue
    t0 = time.time(); te, probs = run_fold(f)
    d = pd.DataFrame({"y": y[te], "pred": probs.argmax(1), "aug": M["aug"].values[te], "pkey": M["pkey"].values[te]})
    for c in range(7): d[f"p{c}"] = probs[:, c]
    d.to_csv(pp, index=False)
    print(f"✅ fold {f} শেষ ({(time.time()-t0)/60:.1f} মিনিট)")

# ---------- মূল্যায়ন ও সেভ (এ পর্যন্ত শেষ হওয়া fold মিলিয়ে) ----------
done = [f for f in FOLDS if os.path.exists(os.path.join(RES, f"pred_{TAG}_fold{f}.csv"))]
P = pd.concat([pd.read_csv(os.path.join(RES, f"pred_{TAG}_fold{f}.csv")).assign(fold=f) for f in done])
save_report(P, f"{TAG}: সব test frame", "all")
if SPLIT_MODE == "paper":
    save_report(P[P["aug"] == 0], f"{TAG}: শুধু আসল test frame", "original")

fw = pd.DataFrame([dict(fold=f, **metrics_table(P[P["fold"] == f]["y"].values, P[P["fold"] == f]["pred"].values)[2]) for f in done])
fw.to_csv(os.path.join(RES, f"foldwise_{TAG}.csv"), index=False)
print("\nfold অনুযায়ী:\n" + fw[["fold", "accuracy", "macro_f1", "sensitivity", "specificity", "precision"]].round(4).to_string(index=False))
if len(done) > 1:
    print("গড় ± std:", {k: f"{fw[k].mean():.4f} ± {fw[k].std():.4f}" for k in ["accuracy", "macro_f1", "sensitivity", "specificity", "precision"]})

with open(os.path.join(RES, f"summary_{TAG}.txt"), "w", encoding="utf-8") as fh:
    fh.write(f"split={SPLIT_MODE} | folds={done} | epochs={EPOCHS} | lr={LR} | batch={BATCH} | CCT params={NPARAM}\n\n" + "\n".join(SUMMARY))
print("\n✅ সব ফলাফল সেভ হয়েছে:", RES)

In [ ]:
# Replication results: training curves, overall and per-class metrics, confusion matrix (reads the replication cell's files).
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
RES = "/content/drive/MyDrive/Data/SeedPaper_CCT/results"
TAG = "cct_paper"            # patient mode চালালে "cct_patient"
CLASSES = ["COPD", "Healthy", "Asthma", "Pneumonia", "URTI", "Bronchiectasis", "Bronchiolitis"]

H = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(os.path.join(RES, f"hist_{TAG}_fold*.csv")))])
m = H.groupby("epoch").mean(numeric_only=True)
nf = int(H.groupby("epoch").size().max()); n_ep = int(H["epoch"].max())
P = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(os.path.join(RES, f"pred_{TAG}_fold*.csv")))])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), dpi=150)                  # curves alone, saved for the paper (also shown in the summary figure)
ax[0].plot(m.index, m["train_loss"], label="Training"); ax[0].plot(m.index, m["val_loss"], label="Validation")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].set_title("Loss"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(m.index, 100 * m["train_acc"], label="Training"); ax[1].plot(m.index, 100 * m["val_acc"], label="Validation")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy (%)"); ax[1].set_title("Accuracy"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.suptitle(f"CCT, seed-paper protocol ({TAG}) - mean over {nf} fold(s)")
fig.tight_layout(); fig.savefig(os.path.join(RES, f"curves_{TAG}.png"), bbox_inches="tight"); plt.close(fig)

def table(d):
    cm = confusion_matrix(d["y"], d["pred"], labels=list(range(7))); N = cm.sum(); rows = []
    for c in range(7):
        TP = cm[c, c]; FN = cm[c].sum() - TP; FP = cm[:, c].sum() - TP; TN = N - TP - FN - FP
        se, sp, pr = TP / max(TP + FN, 1), TN / max(TN + FP, 1), TP / max(TP + FP, 1)
        rows.append((se, sp, pr, 2 * pr * se / max(pr + se, 1e-9)))
    t = pd.DataFrame(rows, index=CLASSES, columns=["Sensitivity (recall)", "Specificity", "Precision", "F1"])
    o = {"Accuracy": np.trace(cm) / N, "Macro-F1": t["F1"].mean(), "Sensitivity (recall)": t["Sensitivity (recall)"].mean(),
         "Specificity": t["Specificity"].mean(), "Precision": t["Precision"].mean()}
    return cm, t, o, int(N), float(cm.sum(1).max() / N)

conds = [("All test frames", P)]
has_aug = "aug" in P and (P["aug"] == 1).any()
if has_aug:
    conds.append(("Original (non-augmented) test frames", P[P["aug"] == 0]))
R = {name: table(d) for name, d in conds}
last = conds[-1][0]; cm, tab = R[last][0], R[last][1]

fig = plt.figure(figsize=(17, 10), dpi=150)
gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.28)

a = fig.add_subplot(gs[0, 0])
a.plot(m.index, m["train_loss"], lw=2, label="Training"); a.plot(m.index, m["val_loss"], lw=2, label="Validation")
a.set_xlabel("Epoch"); a.set_ylabel("Loss"); a.set_title("Loss"); a.grid(alpha=0.3); a.legend()

a = fig.add_subplot(gs[0, 1])
a.plot(m.index, 100 * m["train_acc"], lw=2, label="Training"); a.plot(m.index, 100 * m["val_acc"], lw=2, label="Validation")
a.set_xlabel("Epoch"); a.set_ylabel("Accuracy (%)"); a.set_title("Accuracy"); a.grid(alpha=0.3); a.legend()

a = fig.add_subplot(gs[0, 2])
keys = list(R[conds[0][0]][2]); x = np.arange(len(keys)); w = 0.8 / len(conds)
for i, (name, _) in enumerate(conds):
    off = (i - (len(conds) - 1) / 2) * w
    vals = [R[name][2][k] for k in keys]
    a.bar(x + off, vals, w, label=f"{name} (n={R[name][3]})", edgecolor="black", zorder=3)
    for xi, v in zip(x + off, vals):
        a.text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=7.5)
a.set_xticks(x); a.set_xticklabels([k.replace(" (recall)", "\n(recall)") for k in keys], fontsize=8)
a.set_ylim(0, 1.35); a.set_title("Overall test metrics (macro-averaged)")
a.grid(axis="y", alpha=0.3, zorder=0); a.legend(fontsize=7.5, loc="upper center")

a = fig.add_subplot(gs[1, 0:2])
x = np.arange(7); w = 0.2
for i, mt in enumerate(tab.columns):
    a.bar(x + (i - 1.5) * w, tab[mt].values, w, label=mt, edgecolor="black", zorder=3)
a.set_xticks(x); a.set_xticklabels(CLASSES); a.set_ylim(0, 1.15); a.set_ylabel("Score")
a.set_title(f"Per-class metrics - {last}"); a.grid(axis="y", alpha=0.3, zorder=0); a.legend(ncol=4, fontsize=8, loc="upper center")

a = fig.add_subplot(gs[1, 2])
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
a.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
a.set_xticks(range(7)); a.set_xticklabels(CLASSES, rotation=40, ha="right", fontsize=8)
a.set_yticks(range(7)); a.set_yticklabels(CLASSES, fontsize=8)
for i in range(7):
    for j in range(7):
        a.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if cmn[i, j] > 0.5 else "black")
a.set_xlabel("Predicted"); a.set_ylabel("True"); a.set_title(f"Normalized confusion matrix\n{last}", fontsize=9)

mode = TAG.split("_")[-1]; n_all = R[conds[0][0]][3]
note = f"Seed-paper pipeline with CCT (3-channel 64x40 input). Split mode: {mode}. Folds: {nf}. Epochs: {n_ep}. Test frames: {n_all}"
if has_aug:
    n_o = R[last][3]
    note += f" ({n_o} original, {n_all - n_o} augmented). Always-COPD accuracy on original frames: {R[last][4]:.3f}."
if mode == "paper":
    note += ("\nFrame-level random split (as described in the seed paper): test frames may share recordings (and augmented copies) "
             "with training frames, so these numbers do NOT measure generalization to unseen patients.")
else:
    note += "\nPatient-disjoint split: test patients never appear in training; test uses original recordings only."
fig.text(0.5, 0.005, note, ha="center", va="bottom", fontsize=8.5)
fig.suptitle(f"CCT with seed-paper protocol - {mode} split (preliminary)", fontsize=13, y=0.97)
out = os.path.join(RES, f"sir_summary_{TAG}.png"); fig.savefig(out, bbox_inches="tight"); plt.show()

rows = [{"subset": n, "n_frames": R[n][3], **{k: round(float(v), 4) for k, v in R[n][2].items()}} for n, _ in conds]
pd.DataFrame(rows).to_csv(os.path.join(RES, f"overall_metrics_{TAG}.csv"), index=False)
print(pd.DataFrame(rows).to_string(index=False)); print("✅ ছবি সেভ হয়েছে:", out)

In [ ]:
# Setup (GPU, once per session): writes models_v3.py (models, metrics, statistics, XAI helpers), loads every frame to the GPU,
# defines the paper and patient splits. The trials cell needs it.
import os, sys, importlib, numpy as np, pandas as pd, torch, torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Data"; PROJ = os.path.join(BASE, "SeedPaper_CCT")
FEAT = os.path.join(PROJ, "features"); FT = os.path.join(PROJ, "finetune"); os.makedirs(FT, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for _v in ("Xg", "yg", "DATA"):          # free a previous copy (~1 GB on the GPU) before loading again
    globals().pop(_v, None)
if torch.cuda.is_available(): torch.cuda.empty_cache()

SPLIT_MODE = "paper"; FOLD = 0                         # default split and fold (recorded in the trial settings)
CLASSES = ["COPD", "Healthy", "Asthma", "Pneumonia", "URTI", "Bronchiectasis", "Bronchiolitis"]
LABEL_MAP = {c: i for i, c in enumerate(CLASSES)}

# ---------- shared model / metric code -> Drive file models_v3.py (trials, XAI, band and paper-output cells import it) ----------
MODELS_SRC = r'''# models_v3.py : written by the setup cell, imported by the trials, XAI, band and paper-output cells.
# One source of truth for every model, metric and test used in the BSPC experiments.
import copy, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, matthews_corrcoef, balanced_accuracy_score
from scipy.stats import chi2

NC = 7

# ---------------- CCT family ----------------
class ConvTokenizer(nn.Module):
    """Same layers as the original CCT. dw=True swaps the 2nd 3x3 conv for depthwise 3x3 + pointwise 1x1."""
    def __init__(self, c=3, e=64, dw=False):
        super().__init__()
        conv2 = nn.Sequential(nn.Conv2d(e, e, 3, 1, 1, groups=e), nn.Conv2d(e, e, 1)) if dw else nn.Conv2d(e, e, 3, 1, 1)
        self.net = nn.Sequential(nn.Conv2d(c, e, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(3, 2, 1),
                                 conv2, nn.ReLU(), nn.MaxPool2d(3, 2, 1))
    def forward(self, x):
        return self.net(x).flatten(2).transpose(1, 2)              # (B, nf*nt tokens, e), frequency-major

class Block(nn.Module):
    def __init__(self, e, h, r=2, d=0.1):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(e), nn.LayerNorm(e)
        self.attn = nn.MultiheadAttention(e, h, dropout=d, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(e, e * r), nn.GELU(), nn.Dropout(d), nn.Linear(e * r, e), nn.Dropout(d))
        self.keep_attn, self.last_attn = False, None                # keep_attn=True only for XAI / FLOP counting
    def forward(self, x):
        h = self.n1(x)
        if self.keep_attn:
            a, w = self.attn(h, h, h, need_weights=True, average_attn_weights=True); self.last_attn = w
        else:
            a = self.attn(h, h, h, need_weights=False)[0]
        x = x + a
        return x + self.mlp(self.n2(x))

class CCT(nn.Module):
    """Original CCT (state-dict keys unchanged, so trial_06_best.pt loads as is). dw=True gives variant V2."""
    def __init__(self, nc=NC, e=64, layers=4, heads=4, dropout=0.1, dw=False, nf=16, nt=10):
        super().__init__()
        self.nf, self.nt = nf, nt
        self.tok = ConvTokenizer(3, e, dw)
        self.blocks = nn.ModuleList([Block(e, heads, d=dropout) for _ in range(layers)])
        self.pool = nn.Linear(e, 1); self.fc = nn.Linear(e, nc)
    def forward(self, x, perm=None, ret_pool=False):
        z = self.tok(x)
        if perm is not None: z = z[:, perm]
        for b in self.blocks: z = b(z)
        w = torch.softmax(self.pool(z), dim=1)
        out = self.fc((w.transpose(1, 2) @ z).squeeze(1))
        return (out, w.squeeze(-1)) if ret_pool else out

class FreqCCT(nn.Module):
    """pe: learnable positional embedding over the nf x nt token grid.
       fp: frequency-preserving pooling (LayerNorm, mean over time, keep the nf rows, linear nf*e -> 7)."""
    def __init__(self, nc=NC, e=64, layers=4, heads=4, dropout=0.1, dw=True, pe=True, fp=True, nf=16, nt=10):
        super().__init__()
        self.nf, self.nt, self.fp = nf, nt, fp
        self.tok = ConvTokenizer(3, e, dw)
        if pe:
            self.pos = nn.Parameter(torch.zeros(1, nf * nt, e)); nn.init.trunc_normal_(self.pos, std=0.02)
        else:
            self.pos = None
        self.blocks = nn.ModuleList([Block(e, heads, d=dropout) for _ in range(layers)])
        if fp:
            self.norm = nn.LayerNorm(e); self.fc = nn.Linear(nf * e, nc)
        else:
            self.pool = nn.Linear(e, 1); self.fc = nn.Linear(e, nc)
    def forward(self, x, perm=None, ret_pool=False):
        z = self.tok(x)
        if perm is not None: z = z[:, perm]                          # shuffle content BEFORE position is added
        if self.pos is not None: z = z + self.pos
        for b in self.blocks: z = b(z)
        if self.fp:
            h = self.norm(z).view(z.shape[0], self.nf, self.nt, -1).mean(2)
            out, w = self.fc(h.flatten(1)), None
        else:
            w = torch.softmax(self.pool(z), dim=1)
            out = self.fc((w.transpose(1, 2) @ z).squeeze(1)); w = w.squeeze(-1)
        return (out, w) if ret_pool else out

# ---------------- baselines ----------------
class SimpleCNN2(nn.Module):
    def __init__(self, nc=NC, c1=32, c2=64):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(3, c1, 3, 1, 1), nn.BatchNorm2d(c1), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(c1, c2, 3, 1, 1), nn.BatchNorm2d(c2), nn.ReLU(), nn.MaxPool2d(2),
                               nn.AdaptiveAvgPool2d(1))
        self.fc = nn.Linear(c2, nc)
    def forward(self, x):
        return self.fc(self.f(x).flatten(1))

ARCH_NAMES = {"cct": "CCT", "cct_pefp": "CCT + PE + freq. pooling", "cct_dw": "CCT + depthwise tokenizer",
              "freqcct": "FreqCCT", "cnn2": "2-layer CNN", "mnv3s": "MobileNetV3-small",
              "effb0": "EfficientNet-B0", "res18": "ResNet-18"}
CCT_FAMILY = ("cct", "cct_pefp", "cct_dw", "freqcct")

def build_model(cfg, nf=16):
    a = str(cfg.get("arch", "cct")) if pd.notna(cfg.get("arch", "cct")) else "cct"
    e, L, d = int(cfg.get("embed_dim", 64)), int(cfg.get("num_layers", 4)), float(cfg.get("dropout", 0.0))
    if a == "cct":      return CCT(e=e, layers=L, dropout=d, nf=nf)
    if a == "cct_dw":   return CCT(e=e, layers=L, dropout=d, dw=True, nf=nf)
    if a == "cct_pefp": return FreqCCT(e=e, layers=L, dropout=d, dw=False, pe=True, fp=True, nf=nf)
    if a == "freqcct":  return FreqCCT(e=e, layers=L, dropout=d, dw=True, pe=True, fp=True, nf=nf)
    if a == "cnn2":     return SimpleCNN2()
    from torchvision import models as tvm                          # baselines trained from scratch (no ImageNet weights)
    if a == "mnv3s":    return tvm.mobilenet_v3_small(weights=None, num_classes=NC)
    if a == "effb0":    return tvm.efficientnet_b0(weights=None, num_classes=NC)
    if a == "res18":    return tvm.resnet18(weights=None, num_classes=NC)
    raise ValueError(f"unknown arch {a}")

def n_params(m):
    return int(sum(p.numel() for p in m.parameters()))

# ---------------- compute ----------------
def analytic_mflops(arch, nf=16, nt=10, e=64, L=4, r=2, c=3, nc=NC):
    """2 x multiply-accumulates per frame, block by block (attention included). CCT family only."""
    if arch not in CCT_FAMILY: return float("nan")
    H, W = 4 * nf, 4 * nt; n = nf * nt
    dw = arch in ("cct_dw", "freqcct")
    conv1 = H * W * c * e * 9
    conv2 = (H // 2) * (W // 2) * ((e * 9 + e * e) if dw else e * e * 9)
    attn = L * (n * e * 3 * e + 2 * n * n * e + n * e * e)
    mlp = L * (2 * n * e * e * r)
    head = nf * e * nc if arch in ("cct_pefp", "freqcct") else n * e + e * nc
    return 2.0 * (conv1 + conv2 + attn + mlp + head) / 1e6

def counted_mflops(model, shape):
    """FLOPs from torch's FlopCounterMode. Train mode + keep_attn=True force the explicit attention path:
       the fused eval-mode path of nn.MultiheadAttention is not seen by the counter (the cause of the old 77 MFLOPs)."""
    try:
        from torch.utils.flop_counter import FlopCounterMode
        m = copy.deepcopy(model).cpu().train()
        for b in getattr(m, "blocks", []): b.keep_attn = True
        with torch.no_grad(), FlopCounterMode(display=False) as fc:
            m(torch.zeros(1, *shape))
        return fc.get_total_flops() / 1e6
    except Exception as ex:
        print("FLOP counter unavailable:", type(ex).__name__); return float("nan")

@torch.no_grad()
def cpu_latency_ms(model, shape, warm=50, reps=500):
    m = copy.deepcopy(model).cpu().eval(); x = torch.zeros(1, *shape)
    for _ in range(warm): m(x)
    ts = []
    for _ in range(reps):
        t = time.perf_counter(); m(x); ts.append((time.perf_counter() - t) * 1e3)
    return float(np.median(ts)), float(np.percentile(ts, 90))

def peak_activation_kb(model, shape):
    """Theoretical peak memory of one intermediate tensor (fp32, batch 1): the largest output of any leaf layer,
       and for the CCT family also the attention matrix (heads x tokens x tokens), which nn.MultiheadAttention keeps internally."""
    m = copy.deepcopy(model).cpu().eval(); sizes = []
    hs = [mod.register_forward_hook(lambda mod_, i, o: sizes.append(o.numel() if torch.is_tensor(o) else 0))
          for mod in m.modules() if len(list(mod.children())) == 0]
    with torch.no_grad(): m(torch.zeros(1, *shape))
    for h in hs: h.remove()
    big = max(sizes) if sizes else 0
    if hasattr(m, "blocks"):
        n = m.nf * m.nt; big = max(big, m.blocks[0].attn.num_heads * n * n)
    return 4.0 * big / 1e3

# ---------------- metrics and statistics ----------------
def scores(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return dict(accuracy=float((y == p).mean()),
                macro_f1=float(f1_score(y, p, labels=list(range(NC)), average="macro", zero_division=0)),
                bal_acc=float(balanced_accuracy_score(y, p)), mcc=float(matthews_corrcoef(y, p)))

def rec_vote(P, y, rec):
    """Soft voting: average frame probabilities per recording. Returns (P_rec, y_rec, rec_ids)."""
    df = pd.DataFrame(np.asarray(P)); cols = list(df.columns); df["rec"] = np.asarray(rec); df["y"] = np.asarray(y)
    g = df.groupby("rec", sort=True)
    return g[cols].mean().values, g["y"].first().values.astype(int), np.array(list(g.groups.keys()))

def bootstrap_ci(y, p, groups, B=2000, seed=0, alpha=0.05):
    """Percentile CI, resampling whole groups (recordings) with replacement."""
    y, p, groups = np.asarray(y), np.asarray(p), np.asarray(groups)
    ug, inv = np.unique(groups, return_inverse=True)
    members = [np.where(inv == k)[0] for k in range(len(ug))]
    rng = np.random.default_rng(seed); keys = list(scores(y, p).keys()); out = {k: [] for k in keys}
    for _ in range(B):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(ug), len(ug))])
        s = scores(y[idx], p[idx])
        for k in keys: out[k].append(s[k])
    return {k: (float(np.percentile(v, 100 * alpha / 2)), float(np.percentile(v, 100 * (1 - alpha / 2)))) for k, v in out.items()}

def mcnemar(correct_a, correct_b):
    """Continuity-corrected McNemar (Dietterich 1998): chi2 = (|n01 - n10| - 1)^2 / (n01 + n10), 1 df."""
    a, b = np.asarray(correct_a, bool), np.asarray(correct_b, bool)
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    if n01 + n10 == 0: return 0.0, 1.0, n01, n10
    stat = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
    return float(stat), float(chi2.sf(stat, 1)), n01, n10

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (m - r) * p[i])); adj[i] = run
    return adj

def bh_fdr(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p); adj = np.empty(m); prev = 1.0
    for r in range(m - 1, -1, -1):
        i = order[r]; prev = min(prev, p[i] * m / (r + 1)); adj[i] = prev
    return adj

def select_final(S, cand, tie_pp=0.18):
    """Highest validation accuracy among candidate trials; if within tie_pp percentage points, take fewer MFLOPs."""
    c = S[S["trial"].isin(cand)].copy()
    if c.empty: return None
    c["mf"] = [r["mflops"] if ("mflops" in r and pd.notna(r["mflops"])) else analytic_mflops(str(r.get("arch", "cct")), int(r.get("rows", 16)))
               for _, r in c.iterrows()]
    near = c[c["val_acc"] >= c["val_acc"].max() - tie_pp / 100]
    return int(near.sort_values(["mf", "val_acc"], ascending=[True, False]).iloc[0]["trial"])

# ---------------- explanation helpers (token grid nf x nt) ----------------
def gradcam_tokens(model, xb, tgt):
    """Grad-CAM on the last tokenizer stage (B, e, nf, nt) for target classes tgt (B,). Returns (B, nf, nt)."""
    store = {}
    h = model.tok.net.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    with torch.enable_grad():
        z = model(xb); h.remove()
        A = store["a"]; g, = torch.autograd.grad(z.gather(1, tgt[:, None]).sum(), A)
    return F.relu((g.mean((2, 3), keepdim=True) * A).sum(1)).detach()

@torch.no_grad()
def rollout_tokens(model, xb):
    """Attention rollout (Abnar & Zuidema): A = 0.5 W + 0.5 I per layer, multiplied over layers,
       read out with SeqPool weights (CCT) or uniformly over tokens (frequency pooling). Returns (B, nf, nt)."""
    for b in model.blocks: b.keep_attn = True
    _, w = model(xb, ret_pool=True)
    n = model.blocks[0].last_attn.shape[-1]; I = torch.eye(n, device=xb.device)
    R = I.expand(xb.shape[0], n, n).clone()
    for b in model.blocks:
        A = 0.5 * b.last_attn + 0.5 * I; A = A / A.sum(-1, keepdim=True); R = A @ R
    for b in model.blocks: b.keep_attn = False
    if w is None: w = torch.full((xb.shape[0], n), 1.0 / n, device=xb.device)
    return (w[:, None, :] @ R)[:, 0].view(xb.shape[0], model.nf, model.nt)

def token_mask_to_input(mask_tok):
    """(B, nf, nt) token mask -> (B, 1, 4nf, 4nt) input mask."""
    return mask_tok.repeat_interleave(4, 1).repeat_interleave(4, 2)[:, None]

def mel_row_centres(fmax=2000.0, n=64):
    import librosa
    return librosa.mel_frequencies(n_mels=n + 2, fmin=0.0, fmax=fmax)[1:-1]   # centres of the 64 mel filters

def cqt_row_centres(n=64, fmin=32.7, bpo=12):
    return fmin * 2.0 ** (np.arange(n) / bpo)

def token_row_hz(rows=None):
    """Hz range of each kept token row for the mel and CQT channels (MFCC rows have no Hz meaning)."""
    mel, cqt = mel_row_centres(), cqt_row_centres(); rows = list(range(16)) if rows is None else list(rows)
    return pd.DataFrame([dict(token_row=r, mel_lo_hz=round(mel[4 * r]), mel_hi_hz=round(mel[4 * r + 3]),
                              cqt_lo_hz=round(cqt[4 * r], 1), cqt_hi_hz=round(cqt[4 * r + 3], 1)) for r in rows])
'''
with open(os.path.join(PROJ, "models_v3.py"), "w", encoding="utf-8") as f: f.write(MODELS_SRC)
if PROJ not in sys.path: sys.path.insert(0, PROJ)
import models_v3 as mv; importlib.reload(mv)
from models_v3 import CCT, FreqCCT, build_model

# ---------- load data (original + augmented), same order as every earlier run ----------
def load_feature_set(feat_dir):
    pX = [np.load(os.path.join(feat_dir, "feat_orig.npy"))]
    mo = pd.read_csv(os.path.join(feat_dir, "meta_orig.csv")); mo["aug"] = 0; pM = [mo]
    for c in CLASSES[1:]:
        p = os.path.join(feat_dir, f"feat_aug_{c}.npy")
        if os.path.exists(p):
            pX.append(np.load(p)); m = pd.read_csv(os.path.join(feat_dir, f"meta_aug_{c}.csv")); m["aug"] = 1; pM.append(m)
    return np.concatenate(pX), pd.concat(pM, ignore_index=True)

X, M = load_feature_set(FEAT)
y = M["label"].map(LABEL_MAP).values
Xg = torch.from_numpy(X).to(device); yg = torch.from_numpy(y).long().to(device)
del X
DATA = {"orig": Xg}                                   # other feature sets (e.g. "win64") are loaded on demand
def ensure_feat(tag):
    """Load features_<tag>/ (built by the features cell with VARIANT=tag). Same frames, same order, or it stops."""
    if tag in DATA: return True
    d = os.path.join(PROJ, f"features_{tag}")
    if not os.path.exists(os.path.join(d, f"feat_aug_{CLASSES[-1]}.npy")):
        print(f"features_{tag} not complete: run the features cell with VARIANT='{tag}' first"); return False
    Xa, Ma = load_feature_set(d)
    same = len(Ma) == len(M) and (Ma["rec_idx"].values == M["rec_idx"].values).all() and (Ma["label"].values == M["label"].values).all()
    if not same: raise RuntimeError(f"features_{tag} frames differ from features/: split would not match")
    DATA[tag] = torch.from_numpy(Xa).to(device); return True
print(f"total frames: {len(y)} | original {int((M['aug']==0).sum())} + augmented {int((M['aug']==1).sum())}")

# ---------- split (paper = same as the replication run) ----------
_paper_splits = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(len(y)), y))
def split_idx(f, split="paper"):
    """'paper' = stratified 5-fold over pooled frames (seed protocol); 'patient' = existing patient-disjoint fold column,
       test = real frames of patients in fold f, augmented copies follow their source patient."""
    if split == "paper": return _paper_splits[f]
    tr_ = np.where(M["fold"].values != f)[0]
    te_ = np.where((M["fold"].values == f) & (M["aug"].values == 0))[0]
    return tr_, te_
def fold_idx(f, split="paper"):
    """train / validation / test indices of fold f. 'paper': validation = 10% of training frames, stratified, random_state=42 (unchanged).
       'patient': validation = 10% of training PATIENTS (GroupShuffleSplit on pkey, random_state=42), so no patient is in both."""
    tr_, te_ = split_idx(f, split)
    if split == "paper":
        tr_i, va_i = train_test_split(tr_, test_size=0.10, stratify=y[tr_], random_state=42)
    else:
        from sklearn.model_selection import GroupShuffleSplit
        a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42).split(tr_, groups=M["pkey"].values[tr_]))
        tr_i, va_i = tr_[a], tr_[b]
    return tr_i, va_i, te_
T = lambda a: torch.as_tensor(a, device=device)
_tr, _va, _te = fold_idx(FOLD, SPLIT_MODE)
print(f"fold {FOLD} ({SPLIT_MODE}): train {len(_tr)} | val {len(_va)} | test {len(_te)}")
print("CCT parameters:", mv.n_params(CCT()), "| FreqCCT parameters:", mv.n_params(FreqCCT()))

@torch.no_grad()
def predict(model, idx_t, Xs=None, in_rows=None):
    """Softmax probabilities. Xs = feature tensor (default original), in_rows = kept input rows (band selection)."""
    Xs = Xg if Xs is None else Xs; model.eval(); out = []
    for i in range(0, len(idx_t), 2048):
        xb = Xs[idx_t[i:i + 2048]].float()
        if in_rows is not None: xb = xb[:, :, in_rows]
        out.append(F.softmax(model(xb), 1).cpu())
    return torch.cat(out).numpy()

def evaluate(model, idx_t, Xs=None, in_rows=None):
    P = predict(model, idx_t, Xs, in_rows); yt = y[idx_t.cpu().numpy()]
    return float(-np.log(P[np.arange(len(yt)), yt] + 1e-9).mean()), float((P.argmax(1) == yt).mean())

def metrics_table(yy, pp):
    cm = confusion_matrix(yy, pp, labels=list(range(7))); N = cm.sum(); rows = []
    for c, name in enumerate(CLASSES):
        TP = cm[c, c]; FN = cm[c].sum() - TP; FP = cm[:, c].sum() - TP; TN = N - TP - FN - FP
        se, sp, pr = TP / max(TP + FN, 1), TN / max(TN + FP, 1), TP / max(TP + FP, 1)
        rows.append((name, (TP + TN) / N, se, sp, pr, 2 * pr * se / max(pr + se, 1e-9), int(cm[c].sum())))
    t = pd.DataFrame(rows, columns=["class", "acc_1vsRest", "sensitivity", "specificity", "precision", "F1", "n"]).set_index("class")
    o = dict(accuracy=float(np.trace(cm) / N), majority_acc=float(cm.sum(1).max() / N), macro_f1=float(t["F1"].mean()),
             sensitivity=float(t["sensitivity"].mean()), specificity=float(t["specificity"].mean()),
             precision=float(t["precision"].mean()), n=int(N))
    return cm, t, o
print("setup done. Now run the trials cell.")

In [ ]:
# Trials (GPU): trains each trial in RUN once and saves weights, predictions and metrics in finetune_v2/; finished trials are skipped.
# Final model = highest validation accuracy among SELECT_FROM; within TIE_PP -> fewer MFLOPs. Test frames are never used for selection.
if "Xg" not in globals() or "mv" not in globals(): raise RuntimeError("run the setup cell first in this session")
import os, json, copy, time, datetime, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn.functional as F
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
FT = "/content/drive/MyDrive/Data/SeedPaper_CCT/finetune_v2"; os.makedirs(FT, exist_ok=True)   # new folder, old results untouched
MAX_EPOCHS, PATIENCE, MONITOR = 100, 10, "val_loss"   # early stopping on validation loss

# ================= EDIT ONLY THIS BLOCK =================
# new base = best setting of round 1 (trial 11). New knobs: sched, smooth, seed
# BSPC knobs (added; defaults reproduce trials 0-11 exactly): arch, rows, rank_from, feat, fold, init_from
BASE_CFG = dict(lr=0.002, batch_size=128, weight_decay=0.0, dropout=0.0, embed_dim=64, num_layers=4,
                sched="none", smooth=0.0, seed=0,
                arch="cct", rows=16, rank_from=-1, feat="orig", fold=0, init_from=-1, split="paper")
REF = dict(sched="plateau", smooth=0.1)              # trial-6 recipe, used by every BSPC trial
# Set these two AFTER trials 12-14 finish, from the "Final model" line of the report, BEFORE running trials 15+:
FINAL_TRIAL = 14                                     # trial id of the chosen model on fold 0 (6 if the current CCT stays best)
FINAL_ARCH = "freqcct"                               # its arch: "cct" for trial 6, "cct_pefp" (12), "cct_dw" (13), "freqcct" (14)
TRIALS = [                                           # never reorder, only append at the end
    dict(),                                          # 0  baseline (round-1 best, now with 100 epochs / patience 10)
    dict(seed=1),                                    # 1  same setting, different random start (noise check)
    dict(seed=2),                                    # 2  same setting, different random start (noise check)
    dict(sched="plateau"),                           # 3  halve lr when val loss stalls for 3 epochs
    dict(sched="plateau", lr=0.003),                 # 4  higher start lr + plateau decay
    dict(smooth=0.1),                                # 5  label smoothing 0.1
    dict(sched="plateau", smooth=0.1),               # 6  plateau + label smoothing  (best: 96.86% test, reference row A)
    dict(sched="plateau", lr=0.001),                 # 7  lower start lr + plateau decay
    dict(sched="plateau", batch_size=64),            # 8  smaller batch + plateau decay
    dict(num_layers=3),                              # 9  smaller model
    dict(embed_dim=48),                              # 10 smaller model
    dict(sched="plateau", dropout=0.05),             # 11 tiny dropout + plateau decay
    # ---------- BSPC experiments (fold 0, trial-6 recipe) ----------
    dict(**REF, arch="cct_pefp"),                    # 12 V1: + positional embedding + frequency-preserving pooling
    dict(**REF, arch="cct_dw"),                      # 13 V2: depthwise-separable 2nd tokenizer conv
    dict(**REF, arch="freqcct"),                     # 14 V3: FreqCCT = V1 + V2
    dict(**REF, arch=FINAL_ARCH, rows=12, rank_from=FINAL_TRIAL),   # 15 band selection: keep 12 of 16 rows
    dict(**REF, arch=FINAL_ARCH, rows=8, rank_from=FINAL_TRIAL),    # 16 band selection: keep 8 of 16 rows
    dict(**REF, arch="cnn2"),                        # 17 baseline: 2-layer CNN
    dict(**REF, arch="mnv3s"),                       # 18 baseline: MobileNetV3-small
    dict(**REF, arch="effb0"),                       # 19 baseline: EfficientNet-B0
    dict(**REF, arch="res18"),                       # 20 baseline: ResNet-18
    dict(**REF, arch=FINAL_ARCH, feat="win64"),      # 21 window side test (needs the features cell with VARIANT="win64")
    dict(**REF, arch=FINAL_ARCH, fold=1),            # 22 final model, fold 1
    dict(**REF, arch=FINAL_ARCH, fold=2),            # 23 final model, fold 2
    dict(**REF, arch=FINAL_ARCH, fold=3),            # 24 final model, fold 3
    dict(**REF, arch=FINAL_ARCH, fold=4),            # 25 final model, fold 4
    dict(**REF, arch=FINAL_ARCH, rows=12, rank_from=-2),            # 26 optional: 12 RANDOM rows (control for 15)
    dict(**REF, arch=FINAL_ARCH, init_from=6),       # 27 optional: warm start from trial 6 weights
    # ---------- reviewer checks ----------
    dict(**REF, arch=FINAL_ARCH, split="patient", fold=0),   # 28 patient-disjoint check of the final model (secondary result)
    dict(**REF, arch="cnn2", lr=0.001),              # 29 baseline LR check (vs 17)
    dict(**REF, arch="mnv3s", lr=0.001),             # 30 baseline LR check (vs 18)
    dict(**REF, arch="effb0", lr=0.001),             # 31 baseline LR check (vs 19)
    dict(**REF, arch="res18", lr=0.001),             # 32 baseline LR check (vs 20)
]
RUN = list(range(15))                                # step 1: trials 12-14 (finished trials are skipped)
# RUN = list(range(26)) + [28, 29, 30, 31, 32]       # step 2: after setting FINAL_TRIAL / FINAL_ARCH -> trials 15-25 and 28-32 (26, 27 optional)
SELECT_FROM = [6, 12, 13, 14]                        # candidates for the final model (fold 0, all rows, original features)
TIE_PP = 0.18                                        # validation seed SD (trials 0-2): closer than this = tie -> fewer MFLOPs
# ========================================================

def changed_text(cfg):
    return ", ".join(f"{k}: {BASE_CFG[k]} -> {cfg[k]}" for k in BASE_CFG if cfg[k] != BASE_CFG[k]) or "(none, baseline)"
def label(txt):
    txt = str(txt); return "baseline" if (not txt.isascii() or txt.startswith("(")) else txt
DEFAULTS = {k: BASE_CFG[k] for k in ("arch", "rows", "rank_from", "feat", "fold", "init_from", "split")}

SUM_CSV = os.path.join(FT, "trials_summary.csv")
done = pd.read_csv(SUM_CSV) if os.path.exists(SUM_CSV) else pd.DataFrame()
done_ids = set(done["trial"].astype(int)) if len(done) else set()

def save_params(tag):
    ids = set(pd.read_csv(SUM_CSV)["trial"].astype(int)) if os.path.exists(SUM_CSV) else set()
    rows = []
    for i, t in enumerate(TRIALS):
        cfg = {**BASE_CFG, **t}
        rows.append(dict(trial=i, **cfg, changed_vs_baseline=label(changed_text(cfg)),
                         status="done" if i in ids else ("running now" if i in RUN else "not run")))
    pt = pd.DataFrame(rows); pt.to_csv(os.path.join(FT, "trial_parameters.csv"), index=False)
    meta = dict(saved_at=datetime.datetime.now().isoformat(timespec="seconds"), stage=tag,
                max_epochs=MAX_EPOCHS, early_stopping_patience=PATIENCE, monitor=MONITOR, split_mode=SPLIT_MODE, fold=FOLD,
                optimizer="AdamW (weight_decay 0 = Adam)", loss="cross-entropy (label smoothing = smooth)",
                scheduler="none, or plateau = ReduceLROnPlateau(factor 0.5, patience 3) on validation loss",
                seed="torch.manual_seed(42 + fold + seed)",
                validation="10% of training part, stratified, random_state=42; best epoch chosen on validation, never on test",
                model_selection=f"highest validation accuracy among trials {SELECT_FROM}; within {TIE_PP} pp -> fewer MFLOPs",
                band_selection="Grad-CAM (true class) of trial rank_from on its validation frames, per-frame normalised, mean per class, mean over classes; top-k token rows kept in frequency order",
                base_config=BASE_CFG, final_trial=FINAL_TRIAL, final_arch=FINAL_ARCH,
                trials=[dict(trial=i, changed=t) for i, t in enumerate(TRIALS)], run_this_time=RUN)
    with open(os.path.join(FT, "finetune_settings.json"), "w") as f: json.dump(meta, f, indent=2)
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    with open(os.path.join(FT, f"finetune_settings_run_{stamp}.json"), "w") as f: json.dump(meta, f, indent=2)
    return pt

print(save_params("before training").to_string(index=False))

def load_trial_model(t):
    """Rebuild trial t from its config and load its saved weights."""
    cfg = {**BASE_CFG, **TRIALS[t]}; rows = trial_rows(cfg)
    m = build_model(cfg, nf=16 if rows is None else len(rows)).to(device)
    m.load_state_dict(torch.load(os.path.join(FT, f"trial_{t:02d}_best.pt"), map_location=device)); m.eval()
    return m, cfg, rows

def trial_rows(cfg):
    """Token rows kept by band selection (None = all 16). Ranking is computed once and saved."""
    k = int(cfg["rows"])
    if k >= 16: return None
    src = int(cfg["rank_from"]); fn = os.path.join(FT, f"rows_k{k:02d}_from{src}.json")
    if os.path.exists(fn): return json.load(open(fn))["rows"]
    if src == -2:                                    # random control
        rows = sorted(np.random.default_rng(int(cfg["seed"])).choice(16, k, replace=False).tolist()); score = None
    else:
        if not os.path.exists(os.path.join(FT, f"trial_{src:02d}_best.pt")): raise RuntimeError(f"rank_from trial {src} not trained yet")
        m, cfg_s, _ = load_trial_model(src); _, va_s, _ = fold_idx(int(cfg_s["fold"]), cfg_s["split"]); Xs = DATA[cfg_s["feat"]]
        per = np.zeros((len(va_s), 16))
        for i in range(0, len(va_s), 256):
            b = T(va_s[i:i + 256]); cam = mv.gradcam_tokens(m, Xs[b].float(), yg[b])          # (B, 16, 10)
            cam = cam / cam.sum((1, 2), keepdim=True).clamp(min=1e-12)
            per[i:i + 256] = cam.sum(2).cpu().numpy()
        yv = y[va_s]; score = np.mean([per[yv == c].mean(0) for c in range(7)], axis=0)       # classes weighted equally
        rows = sorted(np.argsort(-score)[:k].tolist())
        bs = mv.token_row_hz(); bs["score"] = score; bs["kept_k12"] = bs.token_row.isin(sorted(np.argsort(-score)[:12]))
        bs["kept_k8"] = bs.token_row.isin(sorted(np.argsort(-score)[:8])); bs.to_csv(os.path.join(FT, f"band_scores_from_trial_{src:02d}.csv"), index=False)
    json.dump(dict(rows=[int(r) for r in rows], k=k, rank_from=src, score=None if score is None else [float(s) for s in score]), open(fn, "w"), indent=2)
    print(f"    band selection: keep rows {rows}")
    return [int(r) for r in rows]

def in_rows_of(rows):
    return None if rows is None else T(np.concatenate([np.arange(4 * r, 4 * r + 4) for r in rows]))

def train_trial(cfg):
    f = int(cfg["fold"]); torch.manual_seed(42 + f + int(cfg["seed"]))
    Xs = DATA[cfg["feat"]]; tr_i, va_i, te_i = fold_idx(f, cfg["split"])
    if cfg["split"] != "paper":                       # show what the patient-disjoint test fold contains
        cnt = np.bincount(y[te_i], minlength=7); pts = M.iloc[te_i].groupby("label")["pkey"].nunique()
        print("    patient test fold: " + ", ".join(f"{CLASSES[c]} {cnt[c]} frames/{int(pts.get(CLASSES[c], 0))} patients" for c in range(7)))
        if (cnt == 0).any(): print("    ⚠️ some classes have no test frame in this patient fold: macro metrics use only the classes present")
    rows = trial_rows(cfg); ir = in_rows_of(rows)
    model = build_model(cfg, nf=16 if rows is None else len(rows)).to(device)
    if int(cfg["init_from"]) >= 0:                   # warm start: copy every tensor whose name and shape match
        src = torch.load(os.path.join(FT, f"trial_{int(cfg['init_from']):02d}_best.pt"), map_location=device); own = model.state_dict()
        keep = {k: v for k, v in src.items() if k in own and own[k].shape == v.shape}; own.update(keep); model.load_state_dict(own)
        print(f"    warm start: {len(keep)}/{len(own)} tensors copied from trial {int(cfg['init_from'])}")
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3) if cfg["sched"] == "plateau" else None
    tr_t, va_t = T(tr_i), T(va_i)
    best, best_ep, bad, state, hist, stopped = float("inf"), 0, 0, None, [], False
    for ep in range(MAX_EPOCHS):
        t0 = time.time(); model.train()
        perm = tr_t[torch.randperm(len(tr_t), device=device)]; tot, corr = 0.0, 0
        for i in range(0, len(perm), int(cfg["batch_size"])):
            b = perm[i:i + int(cfg["batch_size"])]; xb, yb = Xs[b].float(), yg[b]
            if ir is not None: xb = xb[:, :, ir]
            opt.zero_grad(); out = model(xb)
            loss = F.cross_entropy(out, yb, label_smoothing=cfg["smooth"]); loss.backward(); opt.step()
            tot += loss.item() * len(b); corr += (out.argmax(1) == yb).sum().item()
        vl, va = evaluate(model, va_t, Xs, ir)
        cur_lr = opt.param_groups[0]["lr"]
        if sch is not None: sch.step(vl)
        hist.append(dict(epoch=ep + 1, train_loss=tot / len(perm), train_acc=corr / len(perm), val_loss=vl, val_acc=va, lr=cur_lr))
        key = vl if MONITOR == "val_loss" else -va
        if key < best - 1e-4:
            best, best_ep, bad, state = key, ep + 1, 0, copy.deepcopy(model.state_dict())
        else:
            bad += 1
        if ep == 0 or (ep + 1) % 5 == 0:
            print(f"    epoch {ep+1:3d} ({time.time()-t0:.0f}s) lr {cur_lr:.5f} | train loss {tot/len(perm):.3f} acc {corr/len(perm):.3f} | val loss {vl:.3f} acc {va:.3f}")
        if bad >= PATIENCE:
            stopped = True; print(f"    early stopping at epoch {ep+1} (best epoch {best_ep})"); break
    model.load_state_dict(state)
    return model, pd.DataFrame(hist), best_ep, len(hist), stopped, (Xs, va_i, te_i, rows, ir)

for t in RUN:
    if t in done_ids:
        print(f"trial {t} already done, skipped"); continue
    cfg = {**BASE_CFG, **TRIALS[t]}; changed = changed_text(cfg)
    if cfg["feat"] != "orig" and not ensure_feat(cfg["feat"]):
        print(f"trial {t} skipped: features_{cfg['feat']} missing"); continue
    print(f"\n=== trial {t} | changed: {changed}\n    {cfg}")
    t0 = time.time(); model, hist, best_ep, n_ep, stopped, (Xs, va_i, te_i, rows, ir) = train_trial(cfg)
    torch.save(model.state_dict(), os.path.join(FT, f"trial_{t:02d}_best.pt"))
    hist.to_csv(os.path.join(FT, f"hist_trial_{t:02d}.csv"), index=False)
    nf = 16 if rows is None else len(rows)

    probs = predict(model, T(te_i), Xs, ir); pred = probs.argmax(1); yt = y[te_i]; aug = M["aug"].values[te_i]
    rec = M["rec_idx"].values[te_i]
    pr = pd.DataFrame({"y": yt, "pred": pred, "aug": aug, "pkey": M["pkey"].values[te_i], "rec_idx": rec, "fold": int(cfg["fold"]), "split": cfg["split"]})
    for c in range(7): pr[f"p{c}"] = probs[:, c]
    pr.to_csv(os.path.join(FT, f"pred_trial_{t:02d}.csv"), index=False)
    o_all = metrics_table(yt, pred)[2]
    o_org = metrics_table(yt[aug == 0], pred[aug == 0])[2] if (aug == 0).any() else o_all
    r0 = aug == 0; s_org = mv.scores(yt[r0], pred[r0])
    Pr, yr, _ = mv.rec_vote(probs[r0], yt[r0], rec[r0]); s_rec = mv.scores(yr, Pr.argmax(1))

    h = hist.iloc[best_ep - 1]
    row = dict(trial=t, **cfg, changed_vs_baseline=changed, best_epoch=best_ep, stopped_epoch=n_ep, early_stopped=stopped,
               val_loss=float(h["val_loss"]), val_acc=float(h["val_acc"]),
               test_accuracy=o_all["accuracy"], test_macro_f1=o_all["macro_f1"], test_sensitivity=o_all["sensitivity"],
               test_specificity=o_all["specificity"], test_precision=o_all["precision"], test_n=o_all["n"],
               orig_accuracy=o_org["accuracy"], orig_macro_f1=o_org["macro_f1"], orig_sensitivity=o_org["sensitivity"],
               orig_specificity=o_org["specificity"], orig_precision=o_org["precision"], orig_n=o_org["n"],
               orig_bal_acc=s_org["bal_acc"], orig_mcc=s_org["mcc"],
               rec_accuracy=s_rec["accuracy"], rec_macro_f1=s_rec["macro_f1"], rec_mcc=s_rec["mcc"], rec_n=int(len(yr)),
               kept_rows=json.dumps(rows) if rows is not None else "all",
               params=int(sum(p.numel() for p in model.parameters())),
               mflops=mv.counted_mflops(model, (3, 4 * nf, 40)), mflops_analytic=mv.analytic_mflops(cfg["arch"], nf),
               minutes=round((time.time() - t0) / 60, 1))
    done = pd.concat([done, pd.DataFrame([row])], ignore_index=True); done.to_csv(SUM_CSV, index=False)
    print(f"    done: best epoch {best_ep}/{n_ep} | val loss {row['val_loss']:.4f} acc {row['val_acc']:.4f} | "
          f"TEST acc {o_org['accuracy']:.4f} macro-F1 {o_org['macro_f1']:.4f} MCC {s_org['mcc']:.4f} (real frames) | "
          f"recording vote acc {s_rec['accuracy']:.4f} | {row['params']} params, {row['mflops']:.1f} MFLOPs | {row['minutes']} min")

# ---------- best trial (by validation, never by test) ----------
done = pd.read_csv(SUM_CSV).drop_duplicates("trial", keep="last").sort_values("trial").reset_index(drop=True)
done = done.fillna(DEFAULTS)                                               # trials 0-11 were plain CCT, fold 0, all rows
bi_old = done["val_loss"].idxmin()                                         # previous rule, kept for the record
bt = mv.select_final(done[(done["fold"] == 0) & (done["feat"] == "orig") & (done["rows"] == 16) & (done["split"] == "paper")], SELECT_FROM, TIE_PP)
bt = int(done.loc[bi_old, "trial"]) if bt is None else bt
br = done.set_index("trial").loc[bt]
pr = pd.read_csv(os.path.join(FT, f"pred_trial_{bt:02d}.csv")); pr = pr[pr["aug"] == 0] if (pr["aug"] == 0).any() else pr
cm = metrics_table(pr["y"].values, pr["pred"].values)[0]; cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=150); im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(7)); ax.set_xticklabels(CLASSES, rotation=40, ha="right"); ax.set_yticks(range(7)); ax.set_yticklabels(CLASSES)
for i in range(7):
    for j in range(7): ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Final model: trial {bt} (chosen on validation accuracy), real test frames", fontsize=9)
plt.colorbar(im); fig.tight_layout(); fig.savefig(os.path.join(FT, "confusion_best.png")); plt.close(fig)

# ---------- ONE table: train / val / test (test = real frames only) ----------
H = {int(t): pd.read_csv(os.path.join(FT, f"hist_trial_{int(t):02d}.csv")) for t in done["trial"]
     if os.path.exists(os.path.join(FT, f"hist_trial_{int(t):02d}.csv"))}
S = done.set_index("trial"); ts = [t for t in sorted(S.index) if t in H]
def train_at_best(t):
    h = H[t]; be = int(S.loc[t, "best_epoch"]); r = h[h["epoch"] == be]
    return float(r["train_acc"].iloc[0]) if len(r) else float(h["train_acc"].iloc[be - 1])
opt_col = lambda c, t: S.loc[t, c] if c in S.columns else np.nan
tab = pd.DataFrame({"trial": ts, "changed": [label(S.loc[t, "changed_vs_baseline"]) for t in ts],
    "arch": [S.loc[t, "arch"] for t in ts], "split": [S.loc[t, "split"] for t in ts], "fold": [int(S.loc[t, "fold"]) for t in ts], "feat": [S.loc[t, "feat"] for t in ts],
    "rows": [int(S.loc[t, "rows"]) for t in ts],
    "best_epoch": [int(S.loc[t, "best_epoch"]) for t in ts],
    "train_acc_%": [100 * train_at_best(t) for t in ts], "val_acc_%": [100 * S.loc[t, "val_acc"] for t in ts],
    "TEST_acc_%": [100 * S.loc[t, "orig_accuracy"] for t in ts], "TEST_macro_F1": [S.loc[t, "orig_macro_f1"] for t in ts],
    "TEST_MCC": [opt_col("orig_mcc", t) for t in ts], "REC_acc_%": [100 * opt_col("rec_accuracy", t) for t in ts],
    "TEST_sensitivity": [S.loc[t, "orig_sensitivity"] for t in ts], "TEST_specificity": [S.loc[t, "orig_specificity"] for t in ts],
    "TEST_precision": [S.loc[t, "orig_precision"] for t in ts],
    "params": [opt_col("params", t) for t in ts], "MFLOPs": [opt_col("mflops", t) for t in ts]}).round(3)
tab.to_csv(os.path.join(FT, "train_val_test_accuracy_table.csv"), index=False)
lines = [f"Fine-tuning round 2 + BSPC trials | split={SPLIT_MODE} | max epochs={MAX_EPOCHS} | early stopping patience={PATIENCE} | monitor={MONITOR}",
         f"Base config: {BASE_CFG}", "", tab.to_string(index=False), "",
         f"Final model (highest validation accuracy among {SELECT_FROM}; within {TIE_PP} pp -> fewer MFLOPs): trial {bt} ({label(br['changed_vs_baseline'])})",
         f"  TEST (real frames, n={int(br['orig_n'])}): acc {br['orig_accuracy']:.4f} | macro-F1 {br['orig_macro_f1']:.4f} | sens {br['orig_sensitivity']:.4f} | spec {br['orig_specificity']:.4f} | prec {br['orig_precision']:.4f}",
         f"  previous rule (lowest validation loss) would pick trial {int(done.loc[bi_old, 'trial'])}; not used, because label smoothing raises the loss floor (true-class target 0.914 -> loss >= 0.090 on correct frames).",
         f"  -> set FINAL_TRIAL = {bt} and FINAL_ARCH = '{S.loc[bt, 'arch']}' in the edit block before running trials 15+ (now: {FINAL_TRIAL}, '{FINAL_ARCH}')."]
sd = done[done["trial"].isin([0, 1, 2])]
if len(sd) >= 2:
    lines.append(f"Seed noise (trials 0,1,2 = same setting, different seeds): test acc {100*sd['orig_accuracy'].mean():.2f} +/- {100*sd['orig_accuracy'].std():.2f} %, "
                 f"val acc {100*sd['val_acc'].mean():.2f} +/- {100*sd['val_acc'].std():.2f} %  (differences smaller than this are noise)")
ff = [FINAL_TRIAL, 22, 23, 24, 25]                                        # five folds of the final model
if all(t in S.index for t in ff):
    F5 = S.loc[ff, ["fold", "orig_accuracy", "orig_macro_f1", "orig_mcc", "rec_accuracy", "rec_macro_f1"]]
    F5.to_csv(os.path.join(FT, "five_fold_final.csv"))
    lines += ["", "Final model, 5 folds (real test frames):", F5.round(4).to_string(),
              "  mean +/- SD: " + " | ".join(f"{c} {F5[c].mean():.4f} +/- {F5[c].std():.4f}" for c in F5.columns if c != "fold")]
lines += ["train = training accuracy at the selected epoch (dropout on); val = used for early stopping / selection; TEST = real (non-augmented) frames of the held-out fold, evaluated once.",
          "REC = soft vote over the real test frames of each recording.",
          "Split is frame-level (seed-paper protocol): replication result, NOT an estimate for unseen patients."]
open(os.path.join(FT, "finetune_report.txt"), "w", encoding="utf-8").write("\n".join(lines)); print("\n" + "\n".join(lines))

cols_n = 2; rows_n = (len(ts) + cols_n - 1) // cols_n
fig, axs = plt.subplots(rows_n, cols_n, figsize=(12, 3.9 * rows_n), squeeze=False)
for a in axs.ravel(): a.axis("off")
for i, t in enumerate(ts):
    a = axs[i // cols_n][i % cols_n]; a.axis("on"); h = H[t]
    a.plot(h["epoch"], 100 * h["train_acc"], label="train"); a.plot(h["epoch"], 100 * h["val_acc"], label="validation")
    a.axhline(100 * S.loc[t, "orig_accuracy"], ls=":", c="red", label="test (real frames)")
    a.axvline(int(S.loc[t, "best_epoch"]), ls="--", c="gray", label="best epoch")
    a.set_title(f"trial {t}: {label(S.loc[t, 'changed_vs_baseline'])}", fontsize=9 if len(str(S.loc[t, 'changed_vs_baseline'])) < 60 else 7)
    a.set_xlabel("epoch"); a.set_ylabel("accuracy (%)"); a.grid(alpha=0.3); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(FT, "train_val_test_curves.png"), dpi=150); plt.show()

x = np.arange(len(ts)); w = 0.26
fig, ax = plt.subplots(figsize=(max(8, 1.5 * len(ts) + 3), 4.8))
ax.bar(x - w, tab["train_acc_%"], w, label="train (best epoch)"); ax.bar(x, tab["val_acc_%"], w, label="validation")
ax.bar(x + w, tab["TEST_acc_%"], w, label="test (real frames)")
ax.axhline(83.43, ls="--", c="gray", lw=1, label="always-COPD baseline (83.4%)")
ax.set_xticks(x); ax.set_xticklabels([f"trial {t}\n{S.loc[t, 'arch']}" for t in ts], fontsize=6)
ax.set_ylabel("accuracy (%)"); ax.set_ylim(min(80, float(np.nanmin(tab[["val_acc_%", "TEST_acc_%"]].values)) - 2), 100)
ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=8)
ax.set_title("Train vs validation vs test accuracy per trial")
plt.tight_layout(); plt.savefig(os.path.join(FT, "train_val_test_bar.png"), dpi=150); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), dpi=140)
for t in ts:
    hh = H[t]; lab = f"trial {t}" + (" (final)" if t == bt else ""); be = int(S.loc[t, "best_epoch"])
    ax[0].plot(hh["epoch"], hh["val_loss"], label=lab, lw=2.3 if t == bt else 1.1)
    ax[1].plot(hh["epoch"], 100 * hh["val_acc"], label=lab, lw=2.3 if t == bt else 1.1)
    ax[0].scatter([be], [hh["val_loss"].iloc[be - 1]], s=20); ax[1].scatter([be], [100 * hh["val_acc"].iloc[be - 1]], s=20)
ax[0].set_title("Validation loss"); ax[1].set_title("Validation accuracy (%)")
for a in ax: a.set_xlabel("Epoch"); a.grid(alpha=0.3); a.legend(fontsize=6, ncol=2)
fig.suptitle(f"Fine-tuning: max {MAX_EPOCHS} epochs, early stopping patience {PATIENCE} (dots = selected epoch)")
fig.tight_layout(); fig.savefig(os.path.join(FT, "finetune_curves.png")); plt.show()

save_params("after training")
print("\nAll files saved in:", FT)

In [ ]:
# XAI of one trial (default: final model) on the real test frames of its fold -> results/xai_v3
# IG, rollout, Grad-CAM, occlusion, deletion tests, agreement, channel Shapley, shift/shuffle, evidence clips, ICBHI pointing, sanity check.
import os, sys, json, importlib, numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedKFold, train_test_split

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Data"; PROJ = os.path.join(BASE, "SeedPaper_CCT")
FT2 = os.path.join(PROJ, "finetune_v2")
XAI = os.path.join(PROJ, "results", "xai_v3"); os.makedirs(os.path.join(XAI, "audio"), exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not os.path.exists(os.path.join(PROJ, "models_v3.py")): raise RuntimeError("run the setup cell once: it writes models_v3.py")
if PROJ not in sys.path: sys.path.insert(0, PROJ)
import models_v3 as mv; importlib.reload(mv)

# ================= EDIT ONLY THIS BLOCK =================
TRIAL = 14                       # finetune_v2 trial to explain = final model from the trials cell (6 = current best CCT); CCT-family trials only
REF_TRIAL = 6                    # reference model for the shift / shuffle comparison
N_PER_CLASS, N_SANITY = 20, 40   # frames per class for the maps; frames for the random-model sanity check
N_DEL_PER_CLASS, N_RAND_ORDERS, DEL_STEP = 50, 10, 8   # token deletion test: frames per class, random orders, tokens per step
N_NULL_FRAMES, N_SHUF = 200, 1000                       # agreement null: frames, shuffles per frame
ICBHI_ANN = "auto"               # ICBHI cycle annotations: "auto" = the .txt next to each ICBHI .wav, or a folder holding <recording>.txt
ICBHI_KEY = "icbhi"              # substring (case-insensitive) of the "source" column that marks ICBHI recordings
# ========================================================
CLASSES = ["COPD", "Healthy", "Asthma", "Pneumonia", "URTI", "Bronchiectasis", "Bronchiolitis"]
LABEL_MAP = {c: i for i, c in enumerate(CLASSES)}
rng = np.random.default_rng(0); torch.manual_seed(0)
trap = getattr(np, "trapezoid", None) or np.trapz        # numpy 2 renamed trapz

# ---------- trial configuration (from the trials cell's summary) ----------
S2 = pd.read_csv(os.path.join(FT2, "trials_summary.csv")).drop_duplicates("trial", keep="last").set_index("trial")
def trial_cfg(t):
    r = S2.loc[t]; g = lambda k, d: r[k] if (k in r.index and pd.notna(r[k])) else d
    cfg = dict(arch=g("arch", "cct"), embed_dim=int(g("embed_dim", 64)), num_layers=int(g("num_layers", 4)), dropout=float(g("dropout", 0.0)))
    rows = g("kept_rows", "all"); rows = None if rows == "all" else [int(v) for v in json.loads(rows)]
    return cfg, int(g("fold", 0)), str(g("feat", "orig")), rows
cfg, FOLD, FEATTAG, ROWS = trial_cfg(TRIAL)
nf = 16 if ROWS is None else len(ROWS)
IN_ROWS = np.arange(64) if ROWS is None else np.concatenate([np.arange(4 * r, 4 * r + 4) for r in ROWS])
FEAT = os.path.join(PROJ, "features" if FEATTAG == "orig" else f"features_{FEATTAG}")
print(f"explaining trial {TRIAL}: {cfg} | fold {FOLD} | features {FEATTAG} | token rows {ROWS or 'all 16'}")

# ---------- test-এর আসল frame (training সেলের হুবহু একই split) ----------
mo = pd.read_csv(os.path.join(FEAT, "meta_orig.csv")); n_orig = len(mo); mo["aug"] = 0; parts = [mo]
arrays = [np.load(os.path.join(FEAT, "feat_orig.npy"), mmap_mode="r")]
for c in CLASSES[1:]:
    p = os.path.join(FEAT, f"meta_aug_{c}.csv")
    if os.path.exists(p):
        m = pd.read_csv(p); m["aug"] = 1; parts.append(m); arrays.append(np.load(os.path.join(FEAT, f"feat_aug_{c}.npy"), mmap_mode="r"))
M = pd.concat(parts, ignore_index=True); y_all = M["label"].map(LABEL_MAP).values
offs = np.cumsum([0] + [len(a) for a in arrays])
def frames(idx):
    idx = np.asarray(idx); out = np.empty((len(idx), 3, 64, 40), np.float32); which = np.searchsorted(offs, idx, side="right") - 1
    for k in np.unique(which):
        s = which == k; out[s] = arrays[k][idx[s] - offs[k]]
    return out[:, :, IN_ROWS]
tr_f, te = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(len(y_all)), y_all))[FOLD]
tr_i, _ = train_test_split(tr_f, test_size=0.10, stratify=y_all[tr_f], random_state=42)
te = te[te < n_orig]                                          # আসল frame সবার আগে সাজানো
X_te = torch.from_numpy(frames(te)); y_te = y_all[te]; rec_te = M["rec_idx"].values[te]
MU = torch.from_numpy(frames(rng.choice(tr_i, 4000, replace=False)).mean(0)).to(device)   # training-mean input: fill value and IG baseline
print(f"test-এর আসল frame: {len(te)} | recordings: {len(np.unique(rec_te))}")

# ---------- মডেল ----------
def load_model(t):
    c, _, _, rws = trial_cfg(t)
    m = mv.build_model(c, 16 if rws is None else len(rws)).to(device)
    m.load_state_dict(torch.load(os.path.join(FT2, f"trial_{t:02d}_best.pt"), map_location=device)); return m.eval()
model = load_model(TRIAL)
H_IN = X_te.shape[2]

# ---------- ৪টা XAI পদ্ধতি; প্রতিটা (H freq, 40 time) মানচিত্র, 0-1-এ normalize ----------
def norm(a): return (a - a.min()) / (a.max() - a.min() + 1e-8)
up = lambda t: F.interpolate(t.view(1, 1, *t.shape[-2:]), size=(H_IN, 40), mode="bilinear", align_corners=False)[0, 0]

def ig_map(net, x, tgt, steps=64):                            # ১. Integrated Gradients (baseline = training mean, 64 steps)
    al = torch.linspace(0, 1, steps + 1, device=x.device)[1:].view(-1, 1, 1, 1)
    xs = (MU + al * (x - MU)).requires_grad_(True)
    g, = torch.autograd.grad(net(xs)[:, tgt].sum(), xs)
    att = ((x - MU) * g.mean(0)).detach()
    with torch.no_grad(): gap = float(net(x[None])[0, tgt] - net(MU[None])[0, tgt])
    return att, gap                                           # completeness: att.sum() should be close to gap

def rollout_map(net, x): return norm(up(mv.rollout_tokens(net, x[None])[0]))                      # ২. rollout (class-agnostic)
def gradcam_map(net, x, tgt): return norm(up(mv.gradcam_tokens(net, x[None], torch.tensor([tgt], device=x.device))[0]))   # ৩. Grad-CAM

@torch.no_grad()
def occlusion_map(net, x, tgt, win=(8, 8), stride=(4, 4)):    # ৪. Occlusion (ঢাকা অংশ = training mean)
    p0 = F.softmax(net(x[None]), 1)[0, tgt]; H, W = x.shape[1:]
    pos = [(i, j) for i in range(0, H - win[0] + 1, stride[0]) for j in range(0, W - win[1] + 1, stride[1])]
    xs = x[None].repeat(len(pos), 1, 1, 1)
    for k, (i, j) in enumerate(pos): xs[k, :, i:i + win[0], j:j + win[1]] = MU[:, i:i + win[0], j:j + win[1]]
    drop = p0 - F.softmax(net(xs), 1)[:, tgt]
    heat = torch.zeros(H, W, device=x.device); cnt = torch.zeros(H, W, device=x.device)
    for k, (i, j) in enumerate(pos):
        heat[i:i + win[0], j:j + win[1]] += drop[k]; cnt[i:i + win[0], j:j + win[1]] += 1
    return norm(heat / cnt.clamp(min=1))

METHODS = ["Integrated Gradients", "Attention Rollout", "Grad-CAM", "Occlusion"]
def explain(net, x, tgt):
    ig, gap = ig_map(net, x, tgt)
    return ig, gap, {"Integrated Gradients": norm(ig.abs().sum(0)), "Attention Rollout": rollout_map(net, x),
                     "Grad-CAM": gradcam_map(net, x, tgt), "Occlusion": occlusion_map(net, x, tgt)}

# ---------- সব আসল test frame: prediction + token map (Grad-CAM true class, rollout) ----------
P, G_tok, R_tok = [], [], []
for i in range(0, len(X_te), 256):
    xb = X_te[i:i + 256].to(device); tb = torch.as_tensor(y_te[i:i + 256], device=device)
    with torch.no_grad(): P.append(F.softmax(model(xb), 1).cpu())
    G_tok.append(mv.gradcam_tokens(model, xb, tb).cpu()); R_tok.append(mv.rollout_tokens(model, xb).cpu())
P = torch.cat(P); G_tok = torch.cat(G_tok).numpy(); R_tok = torch.cat(R_tok).numpy(); pred = P.argmax(1).numpy()
np.savez_compressed(os.path.join(XAI, f"token_maps_trial{TRIAL:02d}.npz"), gradcam=G_tok, rollout=R_tok, y=y_te, pred=pred, rec=rec_te, test_idx=te)
print(f"accuracy on these frames: {(pred == y_te).mean():.4f} (must equal the trials cell's real-frame test accuracy)")

# ---------- ঠিকভাবে চেনা frame বাছাই ----------
sel = []
for c in range(7):
    cand = np.where((y_te == c) & (pred == c))[0]
    if len(cand) == 0: print(f"⚠️ {CLASSES[c]}: ঠিকভাবে চেনা frame নেই"); continue
    sel += [(c, int(i)) for i in rng.choice(cand, min(N_PER_CLASS, len(cand)), replace=False)]
print(f"বাছাই: {len(sel)} frame (প্রতি ক্লাসে সর্বোচ্চ {N_PER_CLASS})")

# ---------- সব sample-এ মানচিত্র + pixel deletion test + IG completeness ----------
FRACS = [0.05, 0.10, 0.20, 0.30, 0.50]
drop_rec = {m: {f: [] for f in FRACS} for m in METHODS + ["Random"]}
ig_store, ig_gap, shown, shown_idx = {}, [], {}, {}
def mask_top(x, imp, frac, rand=False):
    k = int(frac * imp.numel())
    idx = torch.randperm(imp.numel(), device=x.device)[:k] if rand else torch.topk(imp.flatten(), k).indices
    mk = torch.ones(imp.numel(), device=x.device); mk[idx] = 0; mk = mk.view(1, *imp.shape)
    return x * mk + MU * (1 - mk)
for c, i in sel:
    x = X_te[i].to(device)
    ig, gap, maps = explain(model, x, c)
    ig_store[(c, i)] = ig.cpu().numpy(); ig_gap.append((float(ig.sum()), gap))
    if c not in shown:
        shown[c] = (x.cpu().numpy(), {k: v.cpu().numpy() for k, v in maps.items()}); shown_idx[c] = i
    with torch.no_grad():
        p0 = F.softmax(model(x[None]), 1)[0, c]
        for f in FRACS:
            batch = torch.stack([mask_top(x, maps[m], f) for m in METHODS] + [mask_top(x, maps["Occlusion"], f, rand=True)])
            pm = F.softmax(model(batch), 1)[:, c]
            for k, name in enumerate(METHODS + ["Random"]):
                drop_rec[name][f].append(float(p0 - pm[k]))
ig_gap = np.array(ig_gap); ig_err = np.abs(ig_gap[:, 0] - ig_gap[:, 1]) / (np.abs(ig_gap[:, 1]) + 1e-8)
print(f"IG completeness: median relative error {np.median(ig_err):.3f} (should be < 0.05; raise steps if not)")

# ---------- ছবি ১: প্রতি রোগের একটা উদাহরণ ----------
cls_shown = sorted(shown)
titles = ["Input (mel channel)"] + METHODS
fig, ax = plt.subplots(len(cls_shown), 5, figsize=(17, 2.5 * len(cls_shown)), dpi=130)
ax = np.atleast_2d(ax)
for r, c in enumerate(cls_shown):
    xin, maps = shown[c]
    ax[r, 0].imshow(xin[0], origin="lower", aspect="auto", cmap="viridis", extent=[0, 5, 0, H_IN])
    ax[r, 0].set_ylabel(f"{CLASSES[c]}\np={float(P[shown_idx[c], c]):.2f}", fontsize=9, fontweight="bold")
    for j, mname in enumerate(METHODS):
        ax[r, j + 1].imshow(maps[mname], origin="lower", aspect="auto", cmap="inferno", vmin=0, vmax=1, extent=[0, 5, 0, H_IN])
    for j in range(5):
        ax[r, j].tick_params(labelsize=7)
        if r == 0: ax[r, j].set_title(titles[j], fontsize=9.5)
        if r == len(cls_shown) - 1: ax[r, j].set_xlabel("Time (s)", fontsize=8)
fig.suptitle(f"XAI on trial {TRIAL} ({cfg['arch']}, fold {FOLD}) - one correctly classified original test frame per class", fontsize=12, y=1.0)
fig.tight_layout(); fig.savefig(os.path.join(XAI, "xai_examples.png"), bbox_inches="tight"); plt.show()

# ---------- ছবি ২: pixel deletion test ----------
fig, a = plt.subplots(figsize=(7.5, 5), dpi=140); rows = []
for name in METHODS + ["Random"]:
    mean = [np.mean(drop_rec[name][f]) for f in FRACS]
    a.plot([100 * f for f in FRACS], mean, "o-", lw=1.5 if name == "Random" else 2, ls="--" if name == "Random" else "-",
           label=name, color="gray" if name == "Random" else None)
    rows.append({"method": name, **{f"drop@{int(100*f)}%": round(m, 4) for f, m in zip(FRACS, mean)}})
a.set_xlabel("Top-importance pixels replaced by training mean (%)"); a.set_ylabel("Drop in predicted-class probability")
a.set_title(f"Pixel deletion test (n={len(sel)} frames): above Random = more faithful"); a.grid(alpha=0.3); a.legend()
fig.tight_layout(); fig.savefig(os.path.join(XAI, "xai_deletion.png"), bbox_inches="tight"); plt.show()
pd.DataFrame(rows).to_csv(os.path.join(XAI, "xai_deletion.csv"), index=False)
print(pd.DataFrame(rows).to_string(index=False))

# ---------- A. rollout vs Grad-CAM agreement on the token grid ----------
ntok = nf * 10; ktop = max(1, int(round(0.1 * ntok)))
rho = np.array([spearmanr(R_tok[i].ravel(), G_tok[i].ravel())[0] for i in range(len(te))]); rho = np.nan_to_num(rho)
def topk(a): return set(np.argsort(-a.ravel())[:ktop])
iou = np.array([len(topk(R_tok[i]) & topk(G_tok[i])) / len(topk(R_tok[i]) | topk(G_tok[i])) for i in range(len(te))])
null = []
for i in rng.choice(len(te), min(N_NULL_FRAMES, len(te)), replace=False):
    rr = np.argsort(np.argsort(R_tok[i].ravel())); g = G_tok[i].ravel()
    for _ in range(N_SHUF // 50):
        gp = np.stack([rng.permutation(g) for _ in range(50)]); gr = np.argsort(np.argsort(gp, 1), 1)
        null += list(np.array([np.corrcoef(rr, q)[0, 1] for q in gr]))
thr = float(np.nanpercentile(null, 95))
agree = pd.DataFrame([dict(cls=CLASSES[c], n=int((y_te == c).sum()), rho_median=np.median(rho[y_te == c]),
                           rho_q1=np.percentile(rho[y_te == c], 25), rho_q3=np.percentile(rho[y_te == c], 75),
                           iou_top_median=np.median(iou[y_te == c]), frac_above_null95=float((rho[y_te == c] > thr).mean()))
                      for c in range(7)] +
                     [dict(cls="all", n=len(te), rho_median=np.median(rho), rho_q1=np.percentile(rho, 25), rho_q3=np.percentile(rho, 75),
                           iou_top_median=np.median(iou), frac_above_null95=float((rho > thr).mean())),
                      dict(cls="correct", n=int((pred == y_te).sum()), rho_median=np.median(rho[pred == y_te]), iou_top_median=np.median(iou[pred == y_te])),
                      dict(cls="wrong", n=int((pred != y_te).sum()), rho_median=np.median(rho[pred != y_te]) if (pred != y_te).any() else np.nan,
                           iou_top_median=np.median(iou[pred != y_te]) if (pred != y_te).any() else np.nan)])
agree["null_rho_95pct"] = thr; agree["top_k_tokens"] = ktop
agree.round(4).to_csv(os.path.join(XAI, "xai_agreement.csv"), index=False); print("\nRollout vs Grad-CAM agreement\n", agree.round(3).to_string(index=False))

# ---------- B. token deletion (Grad-CAM order) vs random order ----------
qs = np.arange(0, ntok // 2 + 1, DEL_STEP); frac_ax = qs / ntok
def del_batch(x, order):
    out = []
    for q in qs:
        mk = torch.ones(ntok, device=device); mk[order[:q]] = 0
        mi = mv.token_mask_to_input(mk.view(1, nf, 10))[0]; out.append(x * mi + MU * (1 - mi))
    return torch.stack(out)
dsel = np.concatenate([rng.choice(np.where(y_te == c)[0], min(N_DEL_PER_CLASS, int((y_te == c).sum())), replace=False) for c in range(7)])
curves_a, curves_r, auc_a, auc_r = [], [], [], []
with torch.no_grad():
    for i in dsel:
        x = X_te[i].to(device); c = int(y_te[i])
        order = torch.as_tensor(np.argsort(-G_tok[i].ravel()).copy(), device=device)
        pa = F.softmax(model(del_batch(x, order)), 1)[:, c].cpu().numpy()
        pr_ = np.mean([F.softmax(model(del_batch(x, torch.randperm(ntok, device=device))), 1)[:, c].cpu().numpy() for _ in range(N_RAND_ORDERS)], 0)
        curves_a.append(pa); curves_r.append(pr_); auc_a.append(trap(pa, frac_ax)); auc_r.append(trap(pr_, frac_ax))
auc_a, auc_r = np.array(auc_a), np.array(auc_r); d_auc = auc_r - auc_a; g_rec = rec_te[dsel]
ug = np.unique(g_rec); mem = [np.where(g_rec == u)[0] for u in ug]
boot = [d_auc[np.concatenate([mem[k] for k in rng.integers(0, len(ug), len(ug))])].mean() for _ in range(2000)]
del_res = dict(n_frames=len(dsel), auc_gradcam=float(auc_a.mean()), auc_random=float(auc_r.mean()), diff_random_minus_gradcam=float(d_auc.mean()),
               diff_ci95=[float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))], frac_frames_gradcam_lower=float((d_auc > 0).mean()))
json.dump(del_res, open(os.path.join(XAI, "xai_token_deletion.json"), "w"), indent=2); print("\nToken deletion:", del_res)
fig, a = plt.subplots(figsize=(6.5, 4.5), dpi=140)
for cur, lab, st in [(curves_a, "Grad-CAM order", "-"), (curves_r, f"random order (mean of {N_RAND_ORDERS})", "--")]:
    cur = np.array(cur); m_, s_ = cur.mean(0), cur.std(0) / np.sqrt(len(cur))
    a.plot(100 * frac_ax, m_, st, label=lab); a.fill_between(100 * frac_ax, m_ - 1.96 * s_, m_ + 1.96 * s_, alpha=0.2)
a.set_xlabel("Tokens replaced by training mean (%)"); a.set_ylabel("True-class probability"); a.grid(alpha=0.3); a.legend()
a.set_title(f"Token deletion, n={len(dsel)} frames: lower curve = more faithful map"); fig.tight_layout()
fig.savefig(os.path.join(XAI, "xai_token_deletion.png"), bbox_inches="tight"); plt.show()

# ---------- C. exact Shapley values of the three channels ----------
CH = ["mel", "CQT", "MFCC"]; coal = {}
with torch.no_grad():
    for s in range(8):
        keep = [k for k in range(3) if (s >> k) & 1]; pp = []
        for i in range(0, len(X_te), 512):
            xb = X_te[i:i + 512].to(device).clone()
            for k in range(3):
                if k not in keep: xb[:, k] = MU[k]
            pp.append(F.softmax(model(xb), 1).cpu())
        pp = torch.cat(pp).numpy(); coal[s] = dict(f1=mv.scores(y_te, pp.argmax(1))["macro_f1"],
                                                   cls=[float(pp[y_te == c, c].mean()) for c in range(7)])
def shap(vals, i):
    j, k = [m for m in range(3) if m != i]; b = lambda *m: sum(1 << q for q in m)
    return (vals[b(i)] - vals[0]) / 3 + (vals[b(i, j)] - vals[b(j)]) / 6 + (vals[b(i, k)] - vals[b(k)]) / 6 + (vals[b(i, j, k)] - vals[b(j, k)]) / 3
sh = pd.DataFrame({"channel": CH, "phi_macro_f1": [shap({s: coal[s]["f1"] for s in coal}, i) for i in range(3)]})
for c in range(7): sh[f"phi_p_{CLASSES[c]}"] = [shap({s: coal[s]["cls"][c] for s in coal}, i) for i in range(3)]
sh.round(4).to_csv(os.path.join(XAI, "xai_channel_shapley.csv"), index=False)
print(f"\nChannel Shapley (macro-F1 all={coal[7]['f1']:.4f}, none={coal[0]['f1']:.4f}; sum of phi = difference):\n", sh.round(4).to_string(index=False))
fig, a = plt.subplots(figsize=(5, 3.6), dpi=140); a.bar(CH, sh["phi_macro_f1"], color=["#3b75af", "#ef8636", "#519e3e"])
a.set_ylabel("Shapley value (macro-F1)"); a.set_title("Channel contribution (masked with training mean)", fontsize=9); a.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(XAI, "xai_channel_shapley.png"), bbox_inches="tight"); plt.show()

# ---------- D. property tests on trained models: token shuffle and frequency shift ----------
@torch.no_grad()
def property_tests(net, n=300):
    idx = rng.choice(len(te), min(n, len(te)), replace=False); xb = X_te[idx].to(device); p0 = F.softmax(net(xb), 1)
    sh_ = np.mean([(p0 - F.softmax(net(xb, perm=torch.randperm(net.nf * net.nt, device=device)), 1)).abs().max(1).values.mean().item() for _ in range(5)])
    res = dict(shuffle_mean_max_abs_dprob=float(sh_))
    if net.nf == 16:                                          # same 8-row pattern placed at rows 12-19 vs 36-43 (6 token rows apart)
        xa = MU[None].repeat(len(idx), 1, 1, 1); xb2 = xa.clone(); pat = xb[:, :, 12:20, 10:30]
        xa[:, :, 12:20, 10:30] = pat; xb2[:, :, 36:44, 10:30] = pat
        res["shift_mean_max_abs_dprob"] = float((F.softmax(net(xa), 1) - F.softmax(net(xb2), 1)).abs().max(1).values.mean())
    return res
prop = {f"trial {TRIAL} ({cfg['arch']})": property_tests(model)}
if REF_TRIAL != TRIAL and os.path.exists(os.path.join(FT2, f"trial_{REF_TRIAL:02d}_best.pt")) and trial_cfg(REF_TRIAL)[3] is None:
    prop[f"trial {REF_TRIAL} ({trial_cfg(REF_TRIAL)[0]['arch']})"] = property_tests(load_model(REF_TRIAL))
pd.DataFrame(prop).T.to_csv(os.path.join(XAI, "xai_shift_shuffle.csv")); print("\nShift / shuffle sensitivity\n", pd.DataFrame(prop).T.round(5).to_string())

# ---------- E. listenable evidence clip: time occlusion on the original audio ----------
import librosa, soundfile as sf
SR, FL, HOP = 4000, 20000, 10000
seg = pd.read_csv(os.path.join(BASE, "segment_metadata.csv")); v3 = pd.read_csv(os.path.join(BASE, "final_split_dataset_v3.csv"))
v3r = v3[v3["source"] != "AUGMENTED"][["segment_id", "pkey", "fold"]]
rec_list = (seg.merge(v3r, on="segment_id").drop_duplicates("original_filepath")
               [["original_filepath", "source", "unified_label", "pkey", "fold"]].reset_index(drop=True))   # = the features cell's list
frame_no = mo.groupby("rec_idx").cumcount().values                  # position of each real frame inside its recording
def preprocess_recording(path):                                     # = features cell
    y_, sr = librosa.load(path, sr=None, mono=True)
    if sr != SR: y_ = librosa.resample(y_, orig_sr=sr, target_sr=SR)
    if len(y_) < FL: y_ = np.pad(y_, (0, FL - len(y_)))
    n = 1 + (len(y_) - FL) // HOP; k = int(np.ceil(1.0 * FL / SR)); out = []
    for p in range(n):
        x = y_[p * HOP: p * HOP + FL].astype(np.float64); X = np.fft.rfft(x); X[:k + 1] = 0
        x = np.fft.irfft(X, n=FL); m = np.max(np.abs(x)); out.append((x / m if m > 0 else x).astype(np.float32))
    return np.stack(out)
clips = []
for c in cls_shown:
    i = shown_idx[c]; x = X_te[i].to(device); W4 = 4
    with torch.no_grad():
        p0 = float(F.softmax(model(x[None]), 1)[0, c]); xs = x[None].repeat(40 - W4 + 1, 1, 1, 1)
        for j in range(40 - W4 + 1): xs[j, :, :, j:j + W4] = MU[:, :, j:j + W4]
        drops = p0 - F.softmax(model(xs), 1)[:, c].cpu().numpy()
    j0 = int(np.argmax(drops)); t0s, t1s = j0 * 5 / 40, min(5.0, (j0 + W4) * 5 / 40 + 0.256)   # + one mel window
    gi = int(te[i]); ri = int(M["rec_idx"].values[gi]); path = rec_list.iloc[ri]["original_filepath"]
    try:
        wav = preprocess_recording(path)[frame_no[gi]]
        sf.write(os.path.join(XAI, "audio", f"{CLASSES[c]}_frame.wav"), wav, SR)
        sf.write(os.path.join(XAI, "audio", f"{CLASSES[c]}_evidence_{t0s:.2f}-{t1s:.2f}s.wav"), wav[int(t0s * SR):int(t1s * SR)], SR)
    except Exception as e:
        print(f"⚠️ audio for {CLASSES[c]} not written: {type(e).__name__}")
    clips.append(dict(cls=CLASSES[c], rec_idx=ri, frame_in_recording=int(frame_no[gi]), frame_start_s_in_recording=2.5 * int(frame_no[gi]),
                      evidence_start_s=t0s, evidence_end_s=t1s, prob=p0, max_drop=float(drops[j0])))
pd.DataFrame(clips).to_csv(os.path.join(XAI, "audio", "evidence_clips.csv"), index=False)
print("\nListenable evidence clips\n", pd.DataFrame(clips).round(3).to_string(index=False))

# ---------- F. ICBHI pointing test: does Grad-CAM mass fall in cycles annotated with crackles or wheezes? ----------
# Each ICBHI .txt line: cycle start (s), end (s), crackles (0/1), wheezes (0/1). Token column t of frame k covers [2.5k + 0.5t, 2.5k + 0.5t + 0.5] s.
# Score = share of the frame's Grad-CAM mass (true class) in columns overlapping an adventitious cycle; chance = share of such columns.
def icbhi_txt(path):
    if ICBHI_ANN == "auto": return os.path.splitext(path)[0] + ".txt"
    return os.path.join(ICBHI_ANN, os.path.splitext(os.path.basename(path))[0] + ".txt")
src_te = M["source"].astype(str).str.lower().values[te]
pt_rows, ann_cache = [], {}
for i in np.where(np.char.find(src_te.astype(str), ICBHI_KEY) >= 0)[0]:
    gi = int(te[i]); ri = int(M["rec_idx"].values[gi]); txt = icbhi_txt(rec_list.iloc[ri]["original_filepath"])
    if txt not in ann_cache:
        try: ann_cache[txt] = np.atleast_2d(np.loadtxt(txt))
        except Exception: ann_cache[txt] = None
    ann = ann_cache[txt]
    if ann is None or ann.shape[1] < 4: continue
    s0 = 2.5 * int(frame_no[gi]); cols = np.zeros(10, bool); kind = set()
    for st, en, cr, wh in ann[:, :4]:
        if cr == 0 and wh == 0: continue
        for t in range(10):
            a0, a1 = s0 + 0.5 * t, s0 + 0.5 * t + 0.5
            if min(a1, en) > max(a0, st): cols[t] = True
        if cr: kind.add("crackle")
        if wh: kind.add("wheeze")
    if cols.all() or not cols.any(): continue                       # no contrast inside this frame
    g = G_tok[i].sum(0); g = g / max(g.sum(), 1e-12)
    pt_rows.append(dict(test_pos=i, rec_idx=ri, cls=CLASSES[int(y_te[i])], correct=bool(pred[i] == y_te[i]), kind="+".join(sorted(kind)),
                        score=float(g[cols].sum()), chance=float(cols.mean())))
if pt_rows:
    PT = pd.DataFrame(pt_rows); PT["delta"] = PT["score"] - PT["chance"]
    sg = rng.choice([-1.0, 1.0], size=(10000, len(PT))); dlt = PT["delta"].values
    p_pt = float(((np.abs((sg * dlt).mean(1)) >= abs(dlt.mean())).sum() + 1) / 10001)
    ug = PT["rec_idx"].unique(); mem = [np.where(PT["rec_idx"].values == u)[0] for u in ug]
    bt_ = [dlt[np.concatenate([mem[k] for k in rng.integers(0, len(ug), len(ug))])].mean() for _ in range(2000)]
    summ_pt = dict(n_frames=len(PT), n_recordings=len(ug), mean_score=float(PT["score"].mean()), mean_chance=float(PT["chance"].mean()),
                   mean_delta=float(dlt.mean()), delta_ci95=[float(np.percentile(bt_, 2.5)), float(np.percentile(bt_, 97.5))], p_signflip=p_pt)
    PT.to_csv(os.path.join(XAI, "xai_icbhi_pointing_frames.csv"), index=False)
    PT.groupby(["kind", "correct"])["delta"].agg(["count", "mean"]).round(4).to_csv(os.path.join(XAI, "xai_icbhi_pointing_by_kind.csv"))
    json.dump(summ_pt, open(os.path.join(XAI, "xai_icbhi_pointing.json"), "w"), indent=2)
    print("\nICBHI pointing test (positive delta = attribution concentrates in annotated adventitious cycles):", summ_pt)
else:
    print("\nICBHI pointing test skipped: no ICBHI test frame with annotations found (check ICBHI_ANN / ICBHI_KEY)")

# ---------- sanity check: শেখা মডেল বনাম এলোমেলো (untrained) মডেলের ম্যাপ ----------
torch.manual_seed(0)
rnd = mv.build_model(cfg, nf).to(device).eval()
sc = {m: [] for m in METHODS}
for k in rng.choice(len(sel), min(N_SANITY, len(sel)), replace=False):
    c, i = sel[k]; x = X_te[i].to(device)
    _, _, m_tr = explain(model, x, c); _, _, m_rn = explain(rnd, x, c)
    for m in METHODS:
        r_ = spearmanr(m_tr[m].flatten().cpu().numpy(), m_rn[m].flatten().cpu().numpy())[0]
        sc[m].append(0.0 if np.isnan(r_) else abs(r_))
san = pd.Series({m: float(np.mean(v)) for m, v in sc.items()}, name="mean |Spearman rho| (trained vs random model)")
san.round(3).to_csv(os.path.join(XAI, "xai_sanity.csv"))
print("\nSanity check (low = map depends on the learned weights)\n", san.round(3).to_string())
print("✅ সেভ হয়েছে:", XAI)

In [ ]:
# Band analysis in Hz (run right after the XAI cell): IG band enrichment, clinician view, token-row importance, band-energy test.
if "G_tok" not in globals(): raise RuntimeError("run the XAI cell first in this session")
import librosa

# uses the variables of the XAI cell (run it first): ig_store, sel, shown, shown_idx, P, G_tok, y_te, pred, ROWS, IN_ROWS, nf, XAI, mv
mel_hz = mv.mel_row_centres()[IN_ROWS]                                  # centres of the mel filters for the kept input rows
cqt_hz = mv.cqt_row_centres()[IN_ROWS]                                  # CQT channel-এর সারি -> Hz
EDGES = [0, 100, 300, 600, 1200, 2001]
BN = ["<100", "100-300", "300-600", "600-1200", ">1200"]

def band_vec(att2d, hz):
    """att2d: (rows, 40 time) |attribution|. ফেরত: প্রতি ব্যান্ডে ভাগ আর enrichment (ভাগ / ব্যান্ডের সারির অনুপাত)"""
    row = att2d.sum(1); tot = row.sum() + 1e-12
    frac, enr = [], []
    for lo, hi in zip(EDGES[:-1], EDGES[1:]):
        m = (hz >= lo) & (hz < hi)
        frac.append(row[m].sum() / tot)
        enr.append((row[m].sum() / tot) / m.mean() if m.any() else np.nan)
    return np.array(frac), np.array(enr)

# ---------- ক্লাস অনুযায়ী ব্যান্ড-প্রোফাইল (Integrated Gradients, mel ও CQT channel আলাদা) ----------
prof = {"mel": {}, "CQT": {}}
for c in range(7):
    arrs = [np.abs(ig_store[(cc, i)]) for (cc, i) in sel if cc == c]
    if not arrs: continue
    for name, ch, hz in [("mel", 0, mel_hz), ("CQT", 1, cqt_hz)]:
        prof[name][CLASSES[c]] = np.nanmean(np.stack([band_vec(a[ch], hz)[1] for a in arrs]), axis=0)

fig, axs = plt.subplots(1, 2, figsize=(14, 4.8), dpi=140)
for a, name in zip(axs, ["mel", "CQT"]):
    T_ = pd.DataFrame(prof[name], index=BN).T
    im = a.imshow(T_.values, cmap="RdBu_r", vmin=0, vmax=2, aspect="auto")
    a.set_xticks(range(len(BN))); a.set_xticklabels([b + " Hz" for b in BN], fontsize=8)
    a.set_yticks(range(len(T_))); a.set_yticklabels(T_.index, fontsize=9)
    for r in range(T_.shape[0]):
        for k in range(len(BN)):
            a.text(k, r, f"{T_.values[r, k]:.2f}", ha="center", va="center", fontsize=8)
    a.set_title(f"Integrated Gradients, {name} channel: attribution enrichment per frequency band\n(1.0 = uniform; >1 = model relies more on this band)", fontsize=9)
    plt.colorbar(im, ax=a, fraction=0.04)
    T_.round(3).to_csv(os.path.join(XAI, f"xai_band_enrichment_{name}.csv"))
fig.text(0.5, -0.02,
         "Literature reference ranges: rhonchi < ~300 Hz; coarse crackles ~300-400 Hz; fine crackles ~600-700 Hz (up to ~2 kHz); wheeze dominant >= ~400 Hz; stridor > ~500 Hz.\n"
         "Caveats: frame hop is ~128 ms, so ms-long crackles are not resolved in time; the CQT axis ends at ~1244 Hz (last band has 1 row). A descriptive summary, not diagnostic evidence.",
         ha="center", va="top", fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(XAI, "xai_band_profile.png"), bbox_inches="tight"); plt.show()

# ---------- ডাক্তারের জন্য 'কার্ড': সম্ভাবনা + ম্যাপ + ব্যান্ড ----------
cls_shown = sorted(shown)
rows_hz = {hz: int(np.searchsorted(mel_hz, hz)) for hz in (100, 300, 600, 1200)}
fig, ax = plt.subplots(len(cls_shown), 4, figsize=(17, 2.7 * len(cls_shown)), dpi=130,
                       gridspec_kw={"width_ratios": [1.0, 1.5, 1.5, 1.2]})
ax = np.atleast_2d(ax)
for r, c in enumerate(cls_shown):
    xin, maps = shown[c]; i = shown_idx[c]; prob = np.asarray(P[i]).ravel()
    colors = ["#2ca02c" if k == c else "#9ecae1" for k in range(7)]
    ax[r, 0].barh(range(7), prob, color=colors); ax[r, 0].set_yticks(range(7)); ax[r, 0].set_yticklabels(CLASSES, fontsize=7)
    ax[r, 0].invert_yaxis(); ax[r, 0].set_xlim(0, 1); ax[r, 0].tick_params(labelsize=7)
    ax[r, 0].set_ylabel(f"{CLASSES[c]}\np={prob[c]:.2f}", fontsize=9, fontweight="bold")
    for j, mname in enumerate(["Grad-CAM", "Occlusion"]):
        a = ax[r, j + 1]
        a.imshow(xin[0], origin="lower", aspect="auto", cmap="gray", extent=[0, 5, 0, len(mel_hz)])
        a.imshow(maps[mname], origin="lower", aspect="auto", cmap="jet", alpha=0.45, vmin=0, vmax=1, extent=[0, 5, 0, len(mel_hz)])
        a.set_yticks(list(rows_hz.values())); a.set_yticklabels([f"{h} Hz" for h in rows_hz], fontsize=7); a.tick_params(labelsize=7)
    _, enr = band_vec(np.abs(ig_store[(c, i)][0]), mel_hz)
    ax[r, 3].bar(range(5), enr, color="#fd8d3c"); ax[r, 3].axhline(1, color="k", lw=0.8, ls="--")
    ax[r, 3].set_xticks(range(5)); ax[r, 3].set_xticklabels(BN, fontsize=6.5, rotation=20); ax[r, 3].tick_params(labelsize=7)
    if r == 0:
        for j, t in enumerate(["Model probabilities", "Grad-CAM on mel (Hz axis)", "Occlusion on mel (Hz axis)", "IG enrichment by band (Hz)"]):
            ax[r, j].set_title(t, fontsize=9.5)
    if r == len(cls_shown) - 1:
        for j in (1, 2): ax[r, j].set_xlabel("Time (s)", fontsize=8)
fig.suptitle("Clinician-facing view: what the model relied on (not clinical proof). Check against deletion test and sanity check before use.", fontsize=11, y=1.0)
fig.tight_layout(); fig.savefig(os.path.join(XAI, "xai_clinical_cards.png"), bbox_inches="tight"); plt.show()

# ---------- F. token-row importance (Grad-CAM, all real test frames) with BOTH Hz axes ----------
kept = list(range(16)) if ROWS is None else list(ROWS)
hzt = mv.token_row_hz(kept)
g = G_tok / np.clip(G_tok.sum((1, 2), keepdims=True), 1e-12, None); row_imp = g.sum(2)          # (N, nf), each frame sums to 1
prof_rows = pd.DataFrame({CLASSES[c]: row_imp[y_te == c].mean(0) for c in range(7)})
prof_rows["all classes (equal weight)"] = prof_rows[CLASSES].mean(1)
band_tab = pd.concat([hzt, prof_rows], axis=1); band_tab.round(4).to_csv(os.path.join(XAI, "xai_band_rows_gradcam.csv"), index=False)
ylab = [f"r{int(r.token_row)}: mel {int(r.mel_lo_hz)}-{int(r.mel_hi_hz)} | CQT {r.cqt_lo_hz:.0f}-{r.cqt_hi_hz:.0f} Hz" for r in hzt.itertuples()]
fig, a = plt.subplots(figsize=(10, 0.42 * len(kept) + 1.8), dpi=140)
im = a.imshow(prof_rows.values, aspect="auto", cmap="magma", origin="lower")
a.set_yticks(range(len(kept))); a.set_yticklabels(ylab, fontsize=7); a.set_xticks(range(prof_rows.shape[1]))
a.set_xticklabels(prof_rows.columns, rotation=30, ha="right", fontsize=8); plt.colorbar(im, ax=a, fraction=0.03, label="share of Grad-CAM mass")
a.set_title(f"Token-row importance (trial {TRIAL}); each row covers 4 input rows; MFCC rows have no Hz meaning", fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(XAI, "xai_band_rows_gradcam.png"), bbox_inches="tight"); plt.show()

# ---------- G. attention-guided band-energy test: do the top tokens sit at other frequencies than chance? ----------
mel_c = np.array([mv.mel_row_centres()[4 * r:4 * r + 4].mean() for r in kept]); cqt_c = np.array([mv.cqt_row_centres()[4 * r:4 * r + 4].mean() for r in kept])
ktop = max(1, int(round(0.1 * nf * 10))); tok_mel = np.repeat(mel_c, 10); tok_cqt = np.repeat(cqt_c, 10)
top = np.argsort(-G_tok.reshape(len(G_tok), -1), 1)[:, :ktop]
d_mel = tok_mel[top].mean(1) - tok_mel.mean(); d_cqt = tok_cqt[top].mean(1) - tok_cqt.mean()   # expectation of random tokens = mean of all tokens
def signflip_p(d, n=10000):
    d = d[np.isfinite(d)]; obs = abs(d.mean()); sg = rng.choice([-1.0, 1.0], size=(n, len(d)))
    return float(((np.abs((sg * d).mean(1)) >= obs).sum() + 1) / (n + 1))
bt_rows = [dict(cls=CLASSES[c], n=int((y_te == c).sum()), mean_shift_mel_hz=float(d_mel[y_te == c].mean()), p_mel=signflip_p(d_mel[y_te == c]),
                mean_shift_cqt_hz=float(d_cqt[y_te == c].mean()), p_cqt=signflip_p(d_cqt[y_te == c])) for c in range(7)]
BT = pd.DataFrame(bt_rows); BT["p_mel_fdr"] = mv.bh_fdr(BT["p_mel"]); BT["p_cqt_fdr"] = mv.bh_fdr(BT["p_cqt"])
BT.round(4).to_csv(os.path.join(XAI, "xai_band_energy_test.csv"), index=False)
print(f"Band-energy test: mean frequency of the top-{ktop} Grad-CAM tokens minus the mean of all tokens; sign-flip permutation, BH-FDR across classes")
print(BT.round(4).to_string(index=False))
print("✅ সেভ হয়েছে:", XAI)

In [ ]:
# Paper outputs on a CPU runtime (Runtime -> Change runtime type -> CPU): efficiency, CPU timing, Tables 2 and 6-9, statistics, Figs 3-5.
# Reads saved files of the trials cell only; trains nothing.
import os, sys, json, time, importlib, platform, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt, librosa
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score
from skimage.transform import resize
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Data"; PROJ = os.path.join(BASE, "SeedPaper_CCT")
FT2 = os.path.join(PROJ, "finetune_v2"); FEAT = os.path.join(PROJ, "features")
OUT = os.path.join(PROJ, "bspc_v3"); os.makedirs(OUT, exist_ok=True)
if PROJ not in sys.path: sys.path.insert(0, PROJ)
import models_v3 as mv; importlib.reload(mv)

# ================= EDIT ONLY THIS BLOCK =================
FINAL_TRIAL = 14                      # same as in the trials cell
REF_TRIAL = 6                         # current best CCT (row A)
ABLATION = [6, 12, 13, 14, 15, 16, 21]           # rows A-G of Table 8
BASELINES = [17, 18, 19, 20]
BASELINE_LR = {17: 29, 18: 30, 19: 31, 20: 32}   # lr 0.002 trial -> lr 0.001 trial; the one with higher validation accuracy is reported
PATIENT_TRIAL = 28                    # patient-disjoint check of the final model
FOLD_TRIALS = [FINAL_TRIAL, 22, 23, 24, 25]
B = 2000                              # bootstrap resamples (recordings)
WARM, REPS = 50, 500                  # latency protocol
# ========================================================
CLASSES = ["COPD", "Healthy", "Asthma", "Pneumonia", "URTI", "Bronchiectasis", "Bronchiolitis"]
torch.set_num_threads(1); device = torch.device("cpu")
cpu_name = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), platform.processor()) if os.path.exists("/proc/cpuinfo") else platform.processor()
print(f"CPU: {cpu_name} | torch {torch.__version__} | threads {torch.get_num_threads()} | CUDA visible: {torch.cuda.is_available()}")

S = pd.read_csv(os.path.join(FT2, "trials_summary.csv")).drop_duplicates("trial", keep="last").set_index("trial")
for _c, _d in {"arch": "cct", "rows": 16, "feat": "orig", "fold": 0, "kept_rows": "all", "split": "paper"}.items():   # trials 0-11 lack these columns
    if _c not in S.columns: S[_c] = _d
S = S.fillna({"arch": "cct", "rows": 16, "feat": "orig", "fold": 0, "kept_rows": "all", "split": "paper"})
have = lambda t: t in S.index and os.path.exists(os.path.join(FT2, f"pred_trial_{t:02d}.csv"))
name = lambda t: f"{mv.ARCH_NAMES.get(S.loc[t, 'arch'], S.loc[t, 'arch'])}" + (f" ({int(S.loc[t, 'rows'])} rows)" if int(S.loc[t, "rows"]) < 16 else "") + (f" [{S.loc[t, 'feat']}]" if S.loc[t, "feat"] != "orig" else "")

# ---------- 1. efficiency (single CPU thread, batch 1) ----------
eff = []
for t in sorted(set(ABLATION + BASELINES)):
    if t not in S.index: continue
    r = S.loc[t]; rows = None if r["kept_rows"] == "all" else json.loads(r["kept_rows"]); nf = 16 if rows is None else len(rows)
    cfg = dict(arch=r["arch"], embed_dim=int(r["embed_dim"]), num_layers=int(r["num_layers"]), dropout=float(r["dropout"]))
    m = mv.build_model(cfg, nf); m.load_state_dict(torch.load(os.path.join(FT2, f"trial_{t:02d}_best.pt"), map_location="cpu")); m.eval()
    shape = (3, 4 * nf, 40); med, p90 = mv.cpu_latency_ms(m, shape, WARM, REPS)
    eff.append(dict(trial=t, model=name(t), params=mv.n_params(m), size_mb_fp32=round(mv.n_params(m) * 4 / 1e6, 3),
                    size_kb_int8=round(mv.n_params(m) / 1e3, 1), peak_activation_kb=round(mv.peak_activation_kb(m, shape), 1),   # theoretical memory
                    mflops_counted=round(mv.counted_mflops(m, shape), 2), mflops_analytic=round(mv.analytic_mflops(cfg["arch"], nf), 2),
                    latency_ms_median=round(med, 2), latency_ms_p90=round(p90, 2)))
    print(eff[-1])
EFF = pd.DataFrame(eff, columns=["trial", "model", "params", "size_mb_fp32", "size_kb_int8", "peak_activation_kb", "mflops_counted", "mflops_analytic", "latency_ms_median", "latency_ms_p90"]).set_index("trial")
EFF.to_csv(os.path.join(OUT, "efficiency_cpu.csv"))
bad = EFF[(EFF.mflops_analytic.notna()) & ((EFF.mflops_counted - EFF.mflops_analytic).abs() / EFF.mflops_analytic > 0.02)]
print("FLOP check (counted vs analytic within 2%):", "OK" if bad.empty else f"MISMATCH for trials {list(bad.index)}")

# feature extraction time per 5 s frame (one real frame)
wave = np.load(os.path.join(FEAT, "wave_minority.npy"), mmap_mode="r")[0].astype(np.float32); SR = 4000
def t_of(fn, n=30):
    fn(); ts = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); ts.append((time.perf_counter() - t0) * 1e3)
    return float(np.median(ts)), float(np.percentile(ts, 90))
FEATS = {"mel (256 ms)": lambda: librosa.feature.melspectrogram(y=wave, sr=SR, n_fft=1024, hop_length=524, win_length=1024, window="hamming", n_mels=64),
         "mel (64 ms)": lambda: librosa.feature.melspectrogram(y=wave, sr=SR, n_fft=256, hop_length=524, win_length=256, window="hamming", n_mels=64),
         "CQT": lambda: np.abs(librosa.cqt(wave, sr=SR, hop_length=512, fmin=32.7, n_bins=64, bins_per_octave=12, window="hamming")) ** 2,
         "MFCC (256 ms)": lambda: librosa.feature.mfcc(y=wave, sr=SR, n_mfcc=30, n_mels=30, n_fft=1024, hop_length=512, win_length=1024, window="hamming"),
         "MFCC (64 ms)": lambda: librosa.feature.mfcc(y=wave, sr=SR, n_mfcc=30, n_mels=30, n_fft=256, hop_length=512, win_length=256, window="hamming"),
         "resize to 64x40": lambda: resize(np.random.rand(30, 40), (64, 40), order=1, anti_aliasing=False)}
ftime = pd.DataFrame([dict(component=k, median_ms=v[0], p90_ms=v[1]) for k, v in ((k, t_of(f)) for k, f in FEATS.items())])
if FINAL_TRIAL in EFF.index:
    ftime = pd.concat([ftime, pd.DataFrame([dict(component=f"model: {EFF.loc[FINAL_TRIAL, 'model']}", median_ms=EFF.loc[FINAL_TRIAL, "latency_ms_median"],
                                                 p90_ms=EFF.loc[FINAL_TRIAL, "latency_ms_p90"])])], ignore_index=True)
win = "mel (64 ms)" if (FINAL_TRIAL in S.index and str(S.loc[FINAL_TRIAL, "feat"]) == "win64") else "mel (256 ms)"
used = [win, "CQT", win.replace("mel", "MFCC"), "resize to 64x40"] + [c for c in ftime.component if c.startswith("model")]
ftime["in_end_to_end"] = ftime.component.isin(used); tot = ftime.loc[ftime.in_end_to_end, "median_ms"].sum()
ftime["share_of_end_to_end"] = np.where(ftime.in_end_to_end, ftime.median_ms / tot, np.nan)
ftime.round(3).to_csv(os.path.join(OUT, "table9_cpu_time.csv"), index=False); print("\nTable 9\n", ftime.round(3).to_string(index=False), f"\nend-to-end median {tot:.2f} ms per 5 s frame")

# ---------- 2. predictions: real test frames, recording ids ----------
CL = {c: i for i, c in enumerate(CLASSES)}
mo = pd.read_csv(os.path.join(FEAT, "meta_orig.csv")); parts = [mo.assign(aug=0)]
for c in CLASSES[1:]:
    p = os.path.join(FEAT, f"meta_aug_{c}.csv")
    if os.path.exists(p): parts.append(pd.read_csv(p).assign(aug=1))
M = pd.concat(parts, ignore_index=True); yall = M["label"].map(CL).values
SPL = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(len(yall)), yall))
def preds(t):
    pr = pd.read_csv(os.path.join(FT2, f"pred_trial_{t:02d}.csv"))
    if "rec_idx" not in pr.columns:                       # trials 0-11 saved no rec_idx: rows follow the fold-0 test order
        te = SPL[int(S.loc[t, "fold"])][1]; assert len(te) == len(pr); pr["rec_idx"] = M["rec_idx"].values[te]
    pr = pr[pr["aug"] == 0].reset_index(drop=True); Pm = pr[[f"p{c}" for c in range(7)]].values
    return pr["y"].values, pr["pred"].values, Pm, pr["rec_idx"].values
def both_levels(t):
    yy, pp, Pm, rr = preds(t); Pr, yr, rid = mv.rec_vote(Pm, yy, rr)
    return dict(y=yy, p=pp, P=Pm, rec=rr, yr=yr, pr=Pr.argmax(1), rid=rid)
R = {t: both_levels(t) for t in set(ABLATION + BASELINES + list(BASELINE_LR.values()) + FOLD_TRIALS + [REF_TRIAL, PATIENT_TRIAL]) if have(t)}

# ---------- Table 2: class x dataset (real frames, recordings, patients; augmented copies per class) ----------
_mo = M[M["aug"] == 0]
t2 = _mo.groupby(["source", "label"]).agg(recordings=("rec_idx", "nunique"), patients=("pkey", "nunique"), real_frames=("rec_idx", "size")).reset_index()
t2.to_csv(os.path.join(OUT, "table2_class_by_dataset_long.csv"), index=False)
piv = t2.pivot_table(index="label", columns="source", values="real_frames", fill_value=0).reindex(CLASSES).fillna(0).astype(int)
piv["augmented"] = M[M["aug"] == 1]["label"].value_counts().reindex(CLASSES).fillna(0).astype(int)
piv.to_csv(os.path.join(OUT, "table2_class_by_dataset.csv")); print("\nTable 2 (real frames per dataset; augmented copies per class)\n", piv.to_string())

# ---------- baseline LR check: keep the learning rate with the higher validation accuracy ----------
eff_of, BL, lr_rows = {}, [], []
for a_, b_ in BASELINE_LR.items():
    cand = [t for t in (a_, b_) if t in R]
    if not cand: continue
    pick = max(cand, key=lambda t: S.loc[t, "val_acc"]); BL.append(pick); eff_of[pick] = a_
    lr_rows += [dict(arch=S.loc[t, "arch"], trial=t, lr=S.loc[t, "lr"], val_acc=S.loc[t, "val_acc"], test_acc_real=S.loc[t, "orig_accuracy"], chosen=t == pick) for t in cand]
pd.DataFrame(lr_rows).to_csv(os.path.join(OUT, "table7_baseline_lr_check.csv"), index=False)
if lr_rows: print("\nBaseline LR check (chosen on validation)\n", pd.DataFrame(lr_rows).round(4).to_string(index=False))

def ci_row(t, label):
    d = R[t]; out = dict(model=label, trial=t)
    for lvl, yy, pp, gg in [("frame", d["y"], d["p"], d["rec"]), ("recording", d["yr"], d["pr"], d["rid"])]:
        s = mv.scores(yy, pp); ci = mv.bootstrap_ci(yy, pp, gg, B=B)
        for k in ("accuracy", "macro_f1", "bal_acc", "mcc"):
            out[f"{lvl}_{k}"] = round(s[k], 4); out[f"{lvl}_{k}_ci"] = f"[{ci[k][0]:.4f}, {ci[k][1]:.4f}]"
        out[f"{lvl}_n"] = len(yy)
    return out

# ---------- Table 6: main results ----------
t6 = [ci_row(t, lab) for t, lab in [(FINAL_TRIAL, f"final: {name(FINAL_TRIAL)}"), (REF_TRIAL, f"reference: {name(REF_TRIAL)}")] if t in R]
T6 = pd.DataFrame(t6)
if all(t in R for t in FOLD_TRIALS):
    f5 = pd.DataFrame([dict(fold=int(S.loc[t, "fold"]), **{f"frame_{k}": v for k, v in mv.scores(R[t]["y"], R[t]["p"]).items()},
                            **{f"recording_{k}": v for k, v in mv.scores(R[t]["yr"], R[t]["pr"]).items()}) for t in FOLD_TRIALS])
    f5.round(4).to_csv(os.path.join(OUT, "table6_five_folds.csv"), index=False)
    T6 = pd.concat([T6, pd.DataFrame([dict(model="final, 5-fold mean +/- SD", **{c: f"{f5[c].mean():.4f} +/- {f5[c].std():.4f}" for c in f5.columns if c != "fold"})])], ignore_index=True)
T6.to_csv(os.path.join(OUT, "table6_main.csv"), index=False); print("\nTable 6\n", T6.T.to_string())

# ---------- patient-disjoint check (secondary result) ----------
if PATIENT_TRIAL in R:
    d = R[PATIENT_TRIAL]; pc = ci_row(PATIENT_TRIAL, f"patient-disjoint: {name(PATIENT_TRIAL)}, fold {int(S.loc[PATIENT_TRIAL, 'fold'])}")
    present = np.unique(d["y"]); pc["classes_present"] = len(present)
    pc["frame_macro_f1_present_classes"] = round(float(f1_score(d["y"], d["p"], labels=present, average="macro", zero_division=0)), 4)
    PC = pd.DataFrame([pc]); PC.to_csv(os.path.join(OUT, "table6b_patient_check.csv"), index=False)
    print("\nPatient-disjoint check (frames and recordings of unseen patients)\n", PC.T.to_string())

# ---------- McNemar families (same fold-0 frames) ----------
def mcn_family(ref, others):
    rows = []
    for t in others:
        if t not in R or t == ref: continue
        a, b = R[ref], R[t]
        assert (a["y"] == b["y"]).all(), f"trial {t}: different test frames"
        st, p, n01, n10 = mv.mcnemar(a["p"] == a["y"], b["p"] == b["y"]); str_, pr_, _, _ = mv.mcnemar(a["pr"] == a["yr"], b["pr"] == b["yr"])
        rows.append(dict(trial=t, model=name(t), chi2_frame=st, p_frame=p, only_ref_correct=n01, only_other_correct=n10, chi2_rec=str_, p_rec=pr_))
    df = pd.DataFrame(rows, columns=["trial", "model", "chi2_frame", "p_frame", "only_ref_correct", "only_other_correct", "chi2_rec", "p_rec"])
    if len(df): df["p_frame_holm"] = mv.holm(df.p_frame); df["p_rec_holm"] = mv.holm(df.p_rec)
    return df

# ---------- Table 7: baselines vs final (accuracy and cost) ----------
mc7 = mcn_family(FINAL_TRIAL, BL + [REF_TRIAL, 15, 16]).set_index("trial") if FINAL_TRIAL in R else pd.DataFrame()
t7 = []
for t in BL + [REF_TRIAL, FINAL_TRIAL, 15, 16]:
    if t not in R: continue
    s = mv.scores(R[t]["y"], R[t]["p"]); sr = mv.scores(R[t]["yr"], R[t]["pr"]); e = eff_of.get(t, t)
    row = dict(trial=t, model=name(t), lr=S.loc[t, "lr"], accuracy=s["accuracy"], macro_f1=s["macro_f1"], mcc=s["mcc"], rec_accuracy=sr["accuracy"])
    if e in EFF.index: row.update(EFF.loc[e, ["params", "mflops_counted", "size_kb_int8", "peak_activation_kb", "latency_ms_median", "latency_ms_p90"]].to_dict())
    if t in mc7.index: row.update(chi2=mc7.loc[t, "chi2_frame"], p_holm=mc7.loc[t, "p_frame_holm"], p_rec_holm=mc7.loc[t, "p_rec_holm"])
    t7.append(row)
T7 = pd.DataFrame(t7); T7.round(4).to_csv(os.path.join(OUT, "table7_baselines.csv"), index=False); print("\nTable 7\n", T7.round(4).to_string(index=False))

# ---------- Table 8: ablation, every row vs row A (trial 6) ----------
mc8 = mcn_family(REF_TRIAL, ABLATION).set_index("trial") if REF_TRIAL in R else pd.DataFrame()
t8, base = [], (mv.scores(R[REF_TRIAL]["y"], R[REF_TRIAL]["p"]) if REF_TRIAL in R else None)
for k, t in zip("ABCDEFG", ABLATION):
    if t not in R: continue
    s = mv.scores(R[t]["y"], R[t]["p"])
    row = dict(row=k, trial=t, model=name(t), accuracy=s["accuracy"], macro_f1=s["macro_f1"], mcc=s["mcc"],
               d_acc_pp_vs_A=100 * (s["accuracy"] - base["accuracy"]), d_f1_vs_A=s["macro_f1"] - base["macro_f1"])
    if t in EFF.index: row.update(params=EFF.loc[t, "params"], mflops=EFF.loc[t, "mflops_counted"])
    if t in mc8.index: row.update(p_mcnemar=mc8.loc[t, "p_frame"], p_holm=mc8.loc[t, "p_frame_holm"])
    t8.append(row)
T8 = pd.DataFrame(t8); T8.round(4).to_csv(os.path.join(OUT, "table8_ablation.csv"), index=False)
print("\nTable 8 (seed noise on test accuracy: 0.80 pp)\n", T8.round(4).to_string(index=False))

# ---------- figures ----------
if FINAL_TRIAL in R:                                      # Fig. 3: confusion matrices, frame and recording level
    fig, axs = plt.subplots(1, 2, figsize=(12, 5), dpi=150)
    for a, (yy, pp, ttl) in zip(axs, [(R[FINAL_TRIAL]["y"], R[FINAL_TRIAL]["p"], "frame level"), (R[FINAL_TRIAL]["yr"], R[FINAL_TRIAL]["pr"], "recording level (soft vote)")]):
        cm = confusion_matrix(yy, pp, labels=list(range(7))); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        a.imshow(cmn, cmap="Blues", vmin=0, vmax=1); a.set_xticks(range(7)); a.set_xticklabels(CLASSES, rotation=40, ha="right", fontsize=8)
        a.set_yticks(range(7)); a.set_yticklabels(CLASSES, fontsize=8); a.set_xlabel("Predicted"); a.set_ylabel("True"); a.set_title(f"{ttl}, n={len(yy)}", fontsize=9)
        for i in range(7):
            for j in range(7): a.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=6.5, color="white" if cmn[i, j] > 0.5 else "black")
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig3_confusion.pdf")); fig.savefig(os.path.join(OUT, "fig3_confusion.png")); plt.show()
pts = [(t, EFF.loc[eff_of.get(t, t), "mflops_counted"], mv.scores(R[t]["y"], R[t]["p"])["macro_f1"], EFF.loc[eff_of.get(t, t), "params"])
       for t in set(ABLATION + BL) if t in R and eff_of.get(t, t) in EFF.index]
if pts:                                                   # Fig. 4: macro-F1 vs MFLOPs, marker area ~ parameters
    fig, a = plt.subplots(figsize=(7, 4.8), dpi=150)
    for t, mf, f1, npar in pts:
        a.scatter(mf, f1, s=30 + 300 * npar / max(p[3] for p in pts), alpha=0.7); a.annotate(f"{name(t)}\n{npar/1e3:.0f}k", (mf, f1), fontsize=6.5, xytext=(4, 3), textcoords="offset points")
    a.set_xscale("log"); a.set_xlabel("MFLOPs per 5 s frame (log)"); a.set_ylabel("Macro-F1 (real test frames, fold 0)"); a.grid(alpha=0.3)
    a.set_title("Accuracy against compute (marker size = parameters)", fontsize=9); fig.tight_layout()
    fig.savefig(os.path.join(OUT, "fig4_f1_vs_mflops.pdf")); fig.savefig(os.path.join(OUT, "fig4_f1_vs_mflops.png")); plt.show()
band = [t for t in [FINAL_TRIAL, 15, 16] if t in R and t in EFF.index]
if len(band) >= 2:                                        # Fig. 5: band-selection curve
    fig, a = plt.subplots(figsize=(6.5, 4.5), dpi=150)
    xs_ = [EFF.loc[t, "mflops_counted"] for t in band]
    for key, lab in [("macro_f1", "macro-F1"), ("accuracy", "accuracy")]:
        a.plot(xs_, [mv.scores(R[t]["y"], R[t]["p"])[key] for t in band], "o-", label=lab)
    for t, xv in zip(band, xs_):
        a.annotate(f"{int(S.loc[t, 'rows'])} rows\n{EFF.loc[t, 'latency_ms_median']:.1f} ms", (xv, mv.scores(R[t]["y"], R[t]["p"])["macro_f1"]), fontsize=7, xytext=(4, -18), textcoords="offset points")
    a.set_xlabel("MFLOPs per 5 s frame"); a.set_ylabel("score (real test frames, fold 0)"); a.grid(alpha=0.3); a.legend()
    a.set_title("Band selection: accuracy against compute", fontsize=9); fig.tight_layout()
    fig.savefig(os.path.join(OUT, "fig5_band_selection.pdf")); fig.savefig(os.path.join(OUT, "fig5_band_selection.png")); plt.show()

rep = [f"BSPC summary | CPU {cpu_name} | torch {torch.__version__} | 1 thread, batch 1, {WARM} warm-up + {REPS} timed passes",
       f"final trial {FINAL_TRIAL}, reference trial {REF_TRIAL}; frame-level split (seed-paper protocol); test = real frames",
       "Tables: table6_main.csv, table6_five_folds.csv, table7_baselines.csv, table8_ablation.csv, table9_cpu_time.csv, efficiency_cpu.csv",
       "McNemar: continuity-corrected, Holm-adjusted within each table; CIs: percentile bootstrap over recordings",
       "Also: table2_class_by_dataset.csv, table7_baseline_lr_check.csv, table6b_patient_check.csv; memory: size_kb_int8 = params x 1 byte, peak_activation_kb = largest fp32 tensor at batch 1 (theoretical, not measured on a device)"]
open(os.path.join(OUT, "bspc_v3_report.txt"), "w", encoding="utf-8").write("\n".join(rep))
print("\n".join(rep)); print("✅ সেভ হয়েছে:", OUT)

In [ ]:
# Configuration record: one results/training_config.json with the replication run and the BSPC trials (reads saved files only).
import os, json
import pandas as pd

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Data"; PROJ = os.path.join(BASE, "SeedPaper_CCT")
FEAT, RES, FT2 = os.path.join(PROJ, "features"), os.path.join(PROJ, "results"), os.path.join(PROJ, "finetune_v2")
os.makedirs(RES, exist_ok=True)
CLASSES = ["COPD", "Healthy", "Asthma", "Pneumonia", "URTI", "Bronchiectasis", "Bronchiolitis"]

# ---------- replication run: settings read from the summary the replication cell saved ----------
TAG = "cct_paper"            # patient mode চালালে "cct_patient"
run = dict(split_mode="paper", folds=[0], epochs=20, lr=0.005, batch=128, params=173128)   # used only if the summary file is missing
sp = os.path.join(RES, f"summary_{TAG}.txt")
summary = open(sp, encoding="utf-8").read().splitlines() if os.path.exists(sp) else []
if summary:                  # first line: split=.. | folds=[..] | epochs=.. | lr=.. | batch=.. | CCT params=..
    kv = dict(p.split("=", 1) for p in summary[0].split(" | ") if "=" in p)
    run.update(split_mode=kv.get("split", run["split_mode"]), folds=json.loads(kv.get("folds", "[0]")), epochs=int(kv.get("epochs", run["epochs"])),
               lr=float(kv.get("lr", run["lr"])), batch=int(kv.get("batch", run["batch"])), params=int(kv.get("CCT params", run["params"])))
else:
    print(f"⚠️ {sp} নেই: replication-এর মান ডিফল্ট থেকে নেওয়া হলো")

data = {}
try:
    mo = pd.read_csv(os.path.join(FEAT, "meta_orig.csv"))
    data["original_frames"] = int(len(mo))
    data["original_per_class"] = {k: int(v) for k, v in mo["label"].value_counts().items()}
    aug = {c: int(len(pd.read_csv(os.path.join(FEAT, f"meta_aug_{c}.csv")))) for c in CLASSES[1:]
           if os.path.exists(os.path.join(FEAT, f"meta_aug_{c}.csv"))}
    data["augmented_per_class"] = aug
    data["augmented_frames"] = int(sum(aug.values()))
    data["pooled_frames"] = data["original_frames"] + data["augmented_frames"]
except Exception as e:
    data["note"] = "meta file read failed: " + type(e).__name__

cfg = dict(
    run=dict(tag=TAG, split_mode=run["split_mode"], folds=run["folds"], epochs=run["epochs"],
             optimizer="Adam (PyTorch defaults: betas 0.9/0.999, eps 1e-8, weight_decay 0)",
             learning_rate=run["lr"], lr_schedule="constant (none)",
             batch_size=run["batch"],
             loss="cross-entropy (no class weights, no label smoothing)",
             validation="10% of training part, stratified, random_state=42 (logged only, not used to pick a checkpoint)",
             model_used_for_test="final-epoch model",
             seeds="torch.manual_seed(42 + fold); StratifiedKFold(5, shuffle=True, random_state=42)",
             extras="no PATCH-MIX, no training-time augmentation, no grad clipping, no early stopping, fp32 compute (inputs stored fp16)",
             checkpoint="every 10 epochs (overwritten): checkpoints/ckpt_<TAG>_fold<f>.pt",
             hardware="Colab T4 GPU, about 24-26 s per epoch"),
    model=dict(name="CCT", parameters=run["params"],
               input="(3, 64, 40) = mel, CQT, MFCC stacked",
               tokenizer="2 x [Conv3x3(stride1, pad1, 64ch) + ReLU + MaxPool(3,2,1)] -> 160 tokens (16 freq x 10 time)",
               encoder="4 blocks, pre-LayerNorm, 4 heads, MLP ratio 2 (GELU), dropout 0.1, no positional embedding, no class token",
               pooling="SeqPool (Linear 64->1, softmax over tokens)", classifier="Linear 64->7"),
    data=data,
    preprocessing=dict(sampling_rate_hz=4000, frame_seconds=5, overlap=0.5,
                       baseline_wander="rfft bins 0-5 (0-1 Hz) zeroed", normalization="max-abs",
                       mel="n_fft 1024, hop 524, Hamming, 64 mel, dB (ref=max); 64x39",
                       cqt="64 bins, 12 bins/octave, fmin 32.7 Hz, hop 512, Hamming, power->dB; 64x40",
                       mfcc="30 coeff, 30 mel, n_fft 1024, hop 512, Hamming; 30x40",
                       fusion="per-TFR min-max, bilinear resize to 64x40, stack as 3 channels",
                       augmentation="time-stretch 0.85-1.15, pitch +-3 semitones, noise 0.001-0.01, or combo; balanced up to COPD frame count"),
    caveats=["Frame-level split with augmentation before splitting (seed-paper text): not an estimate for unseen patients.",
             "Augmentation / CQT parameters and dB scaling are our assumptions (not given in the seed paper)."],
)

if summary: cfg["results_summary"] = summary

# ---------- BSPC trials (finetune_v2) ----------
bspc = {}
if os.path.exists(os.path.join(FT2, "finetune_settings.json")):
    bspc["finetune_settings"] = json.load(open(os.path.join(FT2, "finetune_settings.json"), encoding="utf-8"))
if os.path.exists(os.path.join(FT2, "trials_summary.csv")):
    S2 = pd.read_csv(os.path.join(FT2, "trials_summary.csv")).drop_duplicates("trial", keep="last").sort_values("trial")
    keep = [c for c in ["trial", "arch", "rows", "kept_rows", "feat", "fold", "split", "init_from", "lr", "params", "mflops", "mflops_analytic",
                        "best_epoch", "val_acc", "orig_accuracy", "orig_macro_f1", "orig_mcc", "rec_accuracy", "minutes"] if c in S2.columns]
    bspc["trials"] = json.loads(S2[keep].to_json(orient="records"))
bspc["models"] = {"cct": "original CCT (173,128 params): 2-conv tokenizer, 4 pre-LN blocks, 4 heads, MLP ratio 2, SeqPool, no positional embedding",
                   "cct_pefp": "CCT + learnable positional embedding (160 x 64) + frequency-preserving pooling (LayerNorm, mean over time, linear 1024->7)",
                   "cct_dw": "CCT with depthwise 3x3 + pointwise 1x1 as the 2nd tokenizer conv",
                   "freqcct": "cct_pefp + depthwise tokenizer (158,023 params at 16 rows)",
                   "baselines": "2-layer CNN, MobileNetV3-small, EfficientNet-B0, ResNet-18 (torchvision, no pretrained weights), same input, split and recipe"}
bspc["recipe"] = "AdamW wd 0, lr 0.002, batch 128, label smoothing 0.1, ReduceLROnPlateau(0.5, 3) on val loss, max 100 epochs, patience 10, seed 42+fold+seed"
bspc["model_selection"] = "highest validation accuracy among trials 6, 12, 13, 14; ties within 0.18 pp -> fewer MFLOPs (never on test)"
bspc["band_selection"] = "Grad-CAM of the final model on its validation frames, per-frame normalised, class-balanced mean per token row; top 12 / 8 rows kept in frequency order"
bspc["window_test"] = "features_win64: mel and MFCC n_fft = win_length 256 (64 ms), hop unchanged; CQT unchanged and checked identical to features/"
bspc["recording_level"] = "soft vote: mean of frame probabilities over the real test frames of each recording"
bspc["statistics"] = "95% bootstrap CI (2000 resamples of recordings); McNemar with continuity correction; Holm across each comparison family"
bspc["xai"] = "results/xai_v3: IG (training-mean baseline, 64 steps, completeness check), rollout (0.5W+0.5I), Grad-CAM (last tokenizer stage), occlusion, token deletion vs random, exact channel Shapley (8 coalitions, training-mean masking), band-row profile in Hz, band-energy sign-flip test with BH-FDR, shift / shuffle tests, evidence .wav clips"
bspc["timing"] = "paper-output cell: CPU, torch.set_num_threads(1), batch 1, 50 warm-up + 500 timed passes, median and p90; features timed separately"
bspc["patient_check"] = "trial 28: final model on patient fold 0 (existing fold column); validation = 10% of training patients (GroupShuffleSplit on pkey); secondary result"
bspc["baseline_lr_check"] = "trials 29-32: each baseline also at lr 0.001; the lr with the higher validation accuracy is reported"
bspc["icbhi_pointing"] = "share of Grad-CAM mass in token columns overlapping ICBHI cycles annotated with crackles or wheezes vs chance; sign-flip test, bootstrap CI over recordings"
bspc["memory"] = "theoretical: int8 size = parameters x 1 byte; peak activation = largest fp32 tensor at batch 1 (incl. attention matrix); no device measurement"

out = os.path.join(RES, "training_config.json")
with open(out, "w", encoding="utf-8") as f:
    json.dump(dict(replication=cfg, bspc=bspc), f, indent=2, ensure_ascii=False, default=str)
print("✅ সেভ হয়েছে:", out)